# Study of the invariants learned in the latent representations

## Part 0 - Global Configuration

**Library import**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import json
from pathlib import Path
import pickle
from sklearn.manifold import TSNE

**Data Loading**

Chosen variables :

In [ ]:
num_sample = 10000
chosen_autoencoder_type = "CNN" # choose between "MLP" and "CNN"
inv_alignment_method = "swd" # choose between "swd" and "adversarial"
variable = "pr" # variable to predict
val_fraction = 0.1
test_fraction = 0.2
cera_lambda_align = 0.0001
cera_lambda_pred = 0.01

In [ ]:
# Here we import the raw data

precomputed_dir = Path(f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NGS_v{num_sample}")

if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
climate_colors = dict(run_cfg["climate_colors"])
selected_variables_full = list(run_cfg["selected_variables"])
max_abs_lat = float(run_cfg["max_abs_lat"])
patch_size_km = float(run_cfg["patch_size_km"])
time_stride = int(run_cfg["time_stride"])
max_samples_per_climate = int(run_cfg["max_samples_per_climate"])
random_seed = int(run_cfg["random_seed"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_patches = int(run_cfg["n_patches"])

features_by_climate_full = {
    c: np.load(precomputed_dir / f"features_{c}.npy")
    for c in climate_order
}
metadata_by_climate = {
    c: pd.read_csv(precomputed_dir / f"metadata_{c}.csv")
    for c in climate_order
}

# Split off the specified variable as a dedicated label while keeping sample/grid-point alignment.
if variable not in selected_variables_full:
    raise ValueError(f"Variable '{variable}' was not found in run_config selected_variables.")

n_variables_full = len(selected_variables_full)
expected_dim_full = n_variables_full * grid_points_per_patch
variable_var_index = selected_variables_full.index(variable)
variable_col_start = variable_var_index * grid_points_per_patch
variable_col_end = variable_col_start + grid_points_per_patch

feature_mask_without_variable = np.ones(expected_dim_full, dtype=bool)
feature_mask_without_variable[variable_col_start:variable_col_end] = False

label_variable_by_climate = {}
features_by_climate = {}
for c in climate_order:
    X_full = features_by_climate_full[c]
    if X_full.shape[1] != expected_dim_full:
        raise ValueError(
            f"Unexpected feature dimension for {c}: got {X_full.shape[1]}, expected {expected_dim_full}."
        )

    # Keep the specified variable values (one value per patch grid point) as label for each sample.
    label_variable_by_climate[c] = X_full[:, variable_col_start:variable_col_end].copy()

    # Keep all non-variable variables as model features used in the rest of the notebook.
    features_by_climate[c] = X_full[:, feature_mask_without_variable]

selected_variables = [v for v in selected_variables_full if v != variable]

# Convenience aggregate preserving same sample order as stacked climate features.
label_variable = np.vstack([label_variable_by_climate[c] for c in climate_order])

sample_count_df = pd.read_csv(precomputed_dir / "sample_count.csv").set_index("scenario")
pre_sampling_df = pd.read_csv(precomputed_dir / "pre_sampling_df.csv").set_index("scenario")
patch_catalog = pd.read_csv(precomputed_dir / "patch_catalog.csv")
sampling_diagnostics_df = pd.read_csv(precomputed_dir / "sampling_diagnostics.csv").set_index("scenario")
nan_summary_by_variable_df = pd.read_csv(precomputed_dir / "nan_summary_by_variable.csv")
sample_pairs_by_climate = {
    c: pd.read_csv(precomputed_dir / f"sample_pairs_{c}.csv")
    for c in climate_order
}

display(sample_count_df)
display(pre_sampling_df)
display(sampling_diagnostics_df)
print(f"Feature variables used downstream (without {variable}): {selected_variables}")
print(f"label_{variable} shape (all samples x grid points): {label_variable.shape}")

In [ ]:
# Here we import the latent representations for the chosen variables

latent_root = Path("/glade/work/tsalin/CMIP/latent_representations")
if not latent_root.exists():
    raise FileNotFoundError(f"Latent representations directory not found: {latent_root}")

def _load_latent_payload(file_path):
    if not file_path.exists():
        raise FileNotFoundError(f"Latent representations file not found: {file_path}")

    with open(file_path, "rb") as handle:
        payload = pickle.load(handle)

    latent_by_climate = {}
    metadata_by_climate = {}
    for climate, entry in payload.items():
        if "latent" not in entry:
            raise KeyError(f"Missing 'latent' entry for climate '{climate}' in {file_path}")
        latent_by_climate[climate] = np.asarray(entry["latent"])
        metadata_by_climate[climate] = pd.DataFrame(entry["metadata"]).reset_index(drop=True)

    return latent_by_climate, metadata_by_climate

_exp5_latent_dir = latent_root / "latent_representations_exp_5"
_exp3_latent_dir = latent_root / "latent_representations_exp_3"

exp5_cera_latent_file      = _exp5_latent_dir / f"cera_ns{num_sample}_{chosen_autoencoder_type}_{inv_alignment_method}_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_align}_{cera_lambda_pred}_latent_representations_df.pkl"
exp5_baseline_noalign_latent_file = _exp5_latent_dir / f"baseline_CERA_noalign_ns{num_sample}_{chosen_autoencoder_type}_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_pred}_latent_representations_df.pkl"
exp5_cera_full_latent_file       = _exp5_latent_dir / f"cera_full_latent_ns{num_sample}_{chosen_autoencoder_type}_{inv_alignment_method}_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_align}_{cera_lambda_pred}_latent_representations_df.pkl"
exp5_baseline_climax_latent_file = _exp5_latent_dir / f"baseline_ClimaX_ns{num_sample}_{variable}_{val_fraction}_{test_fraction}_latent_representations_df.pkl"
exp3_latent_file_AEall   = _exp3_latent_dir / f"exp3_ns{num_sample}_{chosen_autoencoder_type}_AEall_{variable}_{val_fraction}_{test_fraction}_latent_representations_df.pkl"


exp5_cera_latent_test_by_climate,      exp5_cera_latent_test_metadata_by_climate      = _load_latent_payload(exp5_cera_latent_file)
exp5_baseline_noalign_latent_test_by_climate, exp5_baseline_noalign_latent_test_metadata_by_climate = _load_latent_payload(exp5_baseline_noalign_latent_file)
exp5_cera_full_latent_test_by_climate,    exp5_cera_full_latent_test_metadata_by_climate    = _load_latent_payload(exp5_cera_full_latent_file)
exp5_baseline_climax_latent_test_by_climate, exp5_baseline_climax_latent_test_metadata_by_climate = _load_latent_payload(exp5_baseline_climax_latent_file)
exp3_latent_test_by_climate_AEall, exp3_latent_test_metadata_by_climate_AEall = _load_latent_payload(exp3_latent_file_AEall)

latent_test_sets = {
    "exp5_cera": {
        "latent_test_by_climate": exp5_cera_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp5_cera_latent_test_metadata_by_climate,
        "latent_file": exp5_cera_latent_file,
    },
    "exp5_baseline_noalign": {
        "latent_test_by_climate": exp5_baseline_noalign_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp5_baseline_noalign_latent_test_metadata_by_climate,
        "latent_file": exp5_baseline_noalign_latent_file,
    },
    "exp5_cera_full_latent": {
        "latent_test_by_climate": exp5_cera_full_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp5_cera_full_latent_test_metadata_by_climate,
        "latent_file": exp5_cera_full_latent_file,
    },
    "exp5_baseline_climax": {
        "latent_test_by_climate": exp5_baseline_climax_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp5_baseline_climax_latent_test_metadata_by_climate,
        "latent_file": exp5_baseline_climax_latent_file,
    },
    "exp3_AEall": {
        "latent_test_by_climate": exp3_latent_test_by_climate_AEall,
        "latent_test_metadata_by_climate": exp3_latent_test_metadata_by_climate_AEall,
        "latent_file": exp3_latent_file_AEall,
    },
}

print("Loaded latent representations:")
for latent_name, latent_payload in latent_test_sets.items():
    print(f"  {latent_name}: {latent_payload['latent_file']}")


In [ ]:
# Backward-compatible aliases used later in the notebook
climate_order = list(climate_order)
climate_colors = dict(climate_colors)
sample_count_df = sample_count_df.copy()
pre_sampling_df = pre_sampling_df.copy()
patch_catalog = patch_catalog.copy()
sampling_diagnostics_df = sampling_diagnostics_df.copy()
nan_summary_by_variable_df = nan_summary_by_variable_df.copy()
baseline_train_climates = ["historical"]

## Part 1 - Data preprocessing

Train/test split

In [ ]:
def build_split_indices(data_by_climate, val_fraction=0.10, test_fraction=0.20, seed=42):
    """Build train/val/test split indices for each climate before any standardization."""
    split_indices = {}
    rng = np.random.default_rng(seed)

    for climate, X in data_by_climate.items():
        n = X.shape[0]
        indices = np.arange(n)
        rng.shuffle(indices)

        n_test = max(1, int(round(test_fraction * n)))
        n_val = max(1, int(round(val_fraction * n)))
        n_train = max(1, n - n_val - n_test)

        train_idx = indices[:n_train]
        val_idx = indices[n_train:n_train + n_val]
        test_idx = indices[n_train + n_val:]

        split_indices[climate] = {
            "train": train_idx,
            "val": val_idx,
            "test": test_idx,
        }

    return split_indices


# Build splits on raw, unstandardized data first.
ae_split_indices = build_split_indices(
    features_by_climate,
    val_fraction=val_fraction,
    test_fraction=test_fraction,
    seed=random_seed,
    )

    # ── RAM optimisation ───────
_eval_only_climates = [c for c in climate_order if c not in baseline_train_climates]
if _eval_only_climates:
    for c in _eval_only_climates:
        _idx = ae_split_indices[c]["test"]
        _X_sub = features_by_climate[c][_idx]
        features_by_climate[c] = _X_sub
        _y_sub = label_variable_by_climate[c][_idx]
        label_variable_by_climate[c] = _y_sub
        features_by_climate_full[c] = features_by_climate_full[c][_idx]
        metadata_by_climate[c] = metadata_by_climate[c].iloc[_idx].reset_index(drop=True)
        ae_split_indices[c] = {
            "train": np.array([], dtype=np.intp),
            "val":   np.array([], dtype=np.intp),
            "test":  np.arange(len(_idx), dtype=np.intp),
        }
    print(f"RAM opt : {_eval_only_climates} troncated at {len(_idx)} samples (test split).")
    del _idx, _X_sub, _y_sub
del _eval_only_climates
# ── End of RAM optimisation ──────────────────────────────────────────────────────


# ==============================================================================
# Physical and statistical feature computation (from raw, unscaled data)
# ==============================================================================

n_input_variables = len(selected_variables)


def compute_rh(hus, ta, p):
    eps = 0.622
    e  = (hus * p) / (eps + (1.0 - eps) * hus)
    es = 611.2 * np.exp(17.67 * (ta - 273.15) / (ta - 29.65))
    return e / es


def get_raw_var(X_raw, var_name):
    vi = selected_variables.index(var_name)
    s  = slice(vi * grid_points_per_patch, (vi + 1) * grid_points_per_patch)
    return np.asarray(X_raw[:, s], dtype=np.float64)


# Build monthly-patch climatology from historical train samples only
hist_train_idx  = ae_split_indices["historical"]["train"]
hist_meta_train = metadata_by_climate["historical"].iloc[hist_train_idx].reset_index(drop=True)
X_hist_for_clim = np.asarray(features_by_climate["historical"][hist_train_idx], dtype=np.float64)

from collections import defaultdict

_clim_sums    = defaultdict(lambda: np.zeros(n_input_variables * grid_points_per_patch, dtype=np.float64))
_clim_counts  = defaultdict(int)
_month_sums   = defaultdict(lambda: np.zeros(n_input_variables * grid_points_per_patch, dtype=np.float64))
_month_counts = defaultdict(int)

for _i in range(len(hist_train_idx)):
    _pid = int(hist_meta_train["patch_id"].iloc[_i])
    _m   = int(hist_meta_train["month"].iloc[_i])
    _clim_sums[(_pid, _m)]   += X_hist_for_clim[_i]
    _clim_counts[(_pid, _m)] += 1
    _month_sums[_m]           += X_hist_for_clim[_i]
    _month_counts[_m]         += 1

clim_mean_patch_month = {k: _clim_sums[k] / _clim_counts[k] for k in _clim_sums}
clim_mean_monthly     = {m: _month_sums[m] / _month_counts[m] for m in _month_sums}

del _clim_sums, _clim_counts, _month_sums, _month_counts, X_hist_for_clim
print(f"Climatology: {len(clim_mean_patch_month)} (patch, month) pairs from {len(hist_train_idx)} hist_train samples.")

physical_variable_names = [
    "RH_surface", "RH850", "RH500",
    "lapse_850_500", "surface_850_temp_diff",
    "shear_u", "shear_v", "shear_mag",
    "wind850", "wind500",
] + [f"{v}_anom" for v in selected_variables]
n_physical_variables = len(physical_variable_names)  # 25


def compute_physical_features(X_raw, metadata):
    """Compute physical/statistical features from raw (unscaled) inputs.
    Returns array of shape (n_samples, 25 * grid_points_per_patch).
    Order: RH_surface, RH850, RH500, lapse_850_500, surface_850_temp_diff,
           shear_u, shear_v, shear_mag, wind850, wind500, then 15 anomalies.
    """
    X_raw = np.asarray(X_raw, dtype=np.float64)
    n_s   = X_raw.shape[0]
    gpp   = grid_points_per_patch
    out   = np.empty((n_s, n_physical_variables * gpp), dtype=np.float64)

    tas    = get_raw_var(X_raw, "tas")
    ta850  = get_raw_var(X_raw, "ta850")
    ta500  = get_raw_var(X_raw, "ta500")
    huss   = get_raw_var(X_raw, "huss")
    hus850 = get_raw_var(X_raw, "hus850")
    hus500 = get_raw_var(X_raw, "hus500")
    ua850  = get_raw_var(X_raw, "ua850")
    va850  = get_raw_var(X_raw, "va850")
    ua500  = get_raw_var(X_raw, "ua500")
    va500  = get_raw_var(X_raw, "va500")
    psl    = get_raw_var(X_raw, "psl")

    out[:, 0*gpp:1*gpp]  = compute_rh(huss,   tas,   psl)
    out[:, 1*gpp:2*gpp]  = compute_rh(hus850, ta850, 85000.0)
    out[:, 2*gpp:3*gpp]  = compute_rh(hus500, ta500, 50000.0)
    out[:, 3*gpp:4*gpp]  = ta850 - ta500
    out[:, 4*gpp:5*gpp]  = tas   - ta850
    du = ua500 - ua850
    dv = va500 - va850
    out[:, 5*gpp:6*gpp]  = du
    out[:, 6*gpp:7*gpp]  = dv
    out[:, 7*gpp:8*gpp]  = np.sqrt(du**2 + dv**2)
    out[:, 8*gpp:9*gpp]  = np.sqrt(ua850**2 + va850**2)
    out[:, 9*gpp:10*gpp] = np.sqrt(ua500**2 + va500**2)

    patch_ids  = metadata["patch_id"].values
    months     = metadata["month"].values
    _zero_clim = np.zeros(n_input_variables * gpp, dtype=np.float64)
    for _i in range(n_s):
        _key  = (int(patch_ids[_i]), int(months[_i]))
        _clim = clim_mean_patch_month.get(_key, clim_mean_monthly.get(int(months[_i]), _zero_clim))
        out[_i, 10*gpp:] = X_raw[_i] - _clim

    return out.astype(np.float32)


raw_physical_features_by_climate = {
    c: compute_physical_features(features_by_climate[c], metadata_by_climate[c])
    for c in climate_order
}
print("Raw physical features computed:")
for c in climate_order:
    print(f"  {c}: {raw_physical_features_by_climate[c].shape}")


# ==============================================================================
# Normalization — fit on historical train, apply to all climates
# ==============================================================================

reference_climate = "historical"
if reference_climate not in features_by_climate:
    raise KeyError(f"Reference climate {reference_climate!r} not found in features_by_climate.")

X_hist_train_raw = np.asarray(features_by_climate[reference_climate][hist_train_idx], dtype=np.float32)
y_hist_train_raw = np.asarray(label_variable_by_climate[reference_climate][hist_train_idx], dtype=np.float32)
P_hist_train_raw = np.asarray(raw_physical_features_by_climate[reference_climate][hist_train_idx], dtype=np.float32)

expected_input_dim = n_input_variables * grid_points_per_patch
if X_hist_train_raw.shape[1] != expected_input_dim:
    raise ValueError(
        f"Unexpected input dimension: got {X_hist_train_raw.shape[1]}, "
        f"expected {expected_input_dim} = {n_input_variables} variables x {grid_points_per_patch} points."
    )
if y_hist_train_raw.shape[1] != grid_points_per_patch:
    raise ValueError(
        f"Unexpected label dimension: got {y_hist_train_raw.shape[1]}, expected {grid_points_per_patch}."
    )

# ── Raw input variable normalization ──────────────────────────────────────────
input_variable_means = np.zeros(n_input_variables, dtype=np.float64)
input_variable_stds  = np.ones(n_input_variables,  dtype=np.float64)

for var_idx, var_name in enumerate(selected_variables):
    cols_for_var = slice(
        var_idx * grid_points_per_patch,
        (var_idx + 1) * grid_points_per_patch,
    )
    values = X_hist_train_raw[:, cols_for_var].reshape(-1)
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise ValueError(f"No finite historical train values found for input variable {var_name!r}.")
    if var_name == "pr":
        values = np.log1p(values * 86400)
    mu    = float(np.mean(values))
    sigma = float(np.std(values))
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0
    input_variable_means[var_idx] = mu
    input_variable_stds[var_idx]  = sigma

# ── Label normalization ────────────────────────────────────────────────────────
label_values = y_hist_train_raw.reshape(-1)
label_values = label_values[np.isfinite(label_values)]
if label_values.size == 0:
    raise ValueError(f"No finite historical train values found for label variable {variable!r}.")
if variable == "pr":
    label_values = np.log1p(label_values * 86400)
label_mean = float(np.mean(label_values))
label_std  = float(np.std(label_values))
if not np.isfinite(label_std) or label_std <= 0:
    label_std = 1.0

# ── Physical feature normalization ────────────────────────────────────────────
physical_variable_means = np.zeros(n_physical_variables, dtype=np.float64)
physical_variable_stds  = np.ones(n_physical_variables,  dtype=np.float64)

for phys_idx in range(n_physical_variables):
    cols   = slice(phys_idx * grid_points_per_patch, (phys_idx + 1) * grid_points_per_patch)
    values = P_hist_train_raw[:, cols].reshape(-1)
    values = values[np.isfinite(values)]
    if values.size == 0:
        continue
    mu    = float(np.mean(values))
    sigma = float(np.std(values))
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0
    physical_variable_means[phys_idx] = mu
    physical_variable_stds[phys_idx]  = sigma

# ── Normalization functions ────────────────────────────────────────────────────
def standardize_input_variables(X_raw):
    X_raw    = np.asarray(X_raw, dtype=np.float64)
    X_scaled = np.empty_like(X_raw, dtype=np.float32)
    for var_idx, var_name in enumerate(selected_variables):
        cols_for_var = slice(
            var_idx * grid_points_per_patch,
            (var_idx + 1) * grid_points_per_patch,
        )
        values = X_raw[:, cols_for_var]
        if var_name == "pr":
            values = np.log1p(values * 86400)
        X_scaled[:, cols_for_var] = (
            (values - input_variable_means[var_idx]) / input_variable_stds[var_idx]
        ).astype(np.float32)
    return X_scaled


def standardize_physical_features(P_raw):
    P_raw    = np.asarray(P_raw, dtype=np.float64)
    P_scaled = np.empty_like(P_raw, dtype=np.float32)
    for phys_idx in range(n_physical_variables):
        cols = slice(phys_idx * grid_points_per_patch, (phys_idx + 1) * grid_points_per_patch)
        P_scaled[:, cols] = (
            (P_raw[:, cols] - physical_variable_means[phys_idx]) / physical_variable_stds[phys_idx]
        ).astype(np.float32)
    return P_scaled


def standardize_label_variable(y_raw):
    y_raw = np.asarray(y_raw, dtype=np.float64)
    if variable == "pr":
        y_raw = np.log1p(y_raw * 86400)
    return ((y_raw - label_mean) / label_std).astype(np.float32)


def denormalize_label_variable(y_scaled):
    y_scaled = np.asarray(y_scaled, dtype=np.float64)
    result   = y_scaled * label_std + label_mean
    if variable == "pr":
        result = np.expm1(result)
    return result.astype(np.float32)


# ── Apply normalization and concatenate ───────────────────────────────────────
scaled_features_by_climate          = {}
scaled_label_variable_by_climate    = {}
scaled_physical_features_by_climate = {}
augmented_features_by_climate       = {}

for climate in climate_order:
    X_raw = np.asarray(features_by_climate[climate],              dtype=np.float32)
    y_raw = np.asarray(label_variable_by_climate[climate],        dtype=np.float32)
    P_raw = np.asarray(raw_physical_features_by_climate[climate], dtype=np.float32)

    X_sc = standardize_input_variables(X_raw)
    P_sc = standardize_physical_features(P_raw)

    scaled_features_by_climate[climate]          = X_sc
    scaled_label_variable_by_climate[climate]    = standardize_label_variable(y_raw)
    scaled_physical_features_by_climate[climate] = P_sc
    augmented_features_by_climate[climate]       = np.concatenate([X_sc, P_sc], axis=1)

label_variable_by_climate = scaled_label_variable_by_climate
label_variable = np.vstack([label_variable_by_climate[c] for c in climate_order])

aug_dim = next(iter(augmented_features_by_climate.values())).shape[1]
print(f"\n✓ Augmented feature dimension: {aug_dim}  (1050 raw + {aug_dim - 1050} physical)")
print("Augmented shapes:", {c: augmented_features_by_climate[c].shape for c in climate_order})

In [ ]:
# RAM Reduction
del features_by_climate_full
del features_by_climate
del raw_physical_features_by_climate
del scaled_physical_features_by_climate

## Part 2 - Visualization of invariants in latent representations

t-SNE

In [ ]:
# ── Metadata alignment verification ──────────────────────────────────────────
# For each model (CERA, baselines, exp3) and each climate, check that the latent
# test samples appear in the same order as the raw data at the test-split indices,
# using the compound key (time, patch_id) as a unique sample identifier.
#
# If the order differs (e.g. the training script sorted samples differently),
# a reindex array is computed so that extract_test_latent_and_physical can
# transparently reorder the raw physical features to match the latent order.

def _sample_keys(meta_df):
    """Return an array of unique 'time|patch_id' string keys, one per row."""
    return (meta_df["time"].astype(str) + "|" + meta_df["patch_id"].astype(str)).values


reindex_maps = {} 
_any_reindex = False

for _lkey in latent_test_sets.keys():
    reindex_maps[_lkey] = {}
    for c in climate_order:
        # Keys from the latent payload metadata
        _lmeta     = pd.DataFrame(latent_test_sets[_lkey]["latent_test_metadata_by_climate"][c])
        keys_latent = _sample_keys(_lmeta)

        # Keys from the raw data metadata at the test-split indices
        if c == "historical":
            _test_idx = ae_split_indices["historical"]["test"]
            _raw_meta = metadata_by_climate["historical"].iloc[_test_idx].reset_index(drop=True)
        else:
            # SSP climates were already truncated to their test split by the RAM optimisation
            _raw_meta = metadata_by_climate[c]
        keys_raw = _sample_keys(_raw_meta)

        # ── Size check ────────────────────────────────────────────────────────
        if len(keys_latent) != len(keys_raw):
            raise ValueError(
                f"[{_lkey}/{c}] Size mismatch: latent has {len(keys_latent)} samples "
                f"but raw data at test split has {len(keys_raw)}. "
                "Verify that num_samples, val_fraction, and test_fraction match "
                "the values used when producing the latent representations."
            )

        # ── Order check ────────────────────────────────────────────────────────
        if np.array_equal(keys_latent, keys_raw):
            reindex_maps[_lkey][c] = None   # already aligned — no reindexing needed
        else:
            # Build a position map from the raw keys and compute the reindex array
            _raw_pos = {k: i for i, k in enumerate(keys_raw)}
            _missing  = [k for k in keys_latent if k not in _raw_pos]
            if _missing:
                raise ValueError(
                    f"[{_lkey}/{c}] {len(_missing)} latent samples have no corresponding "
                    "entry in the raw test split. The datasets may not be compatible — "
                    "check num_samples and split fractions."
                )
            _reindex = np.array([_raw_pos[k] for k in keys_latent], dtype=np.intp)
            reindex_maps[_lkey][c] = _reindex
            _any_reindex = True
            print(f"  ⚠  [{_lkey}/{c}] Order mismatch detected — reindex array computed "
                  f"({len(_reindex)} samples).")

if not _any_reindex:
    print("✓  Metadata alignment verified for all models and climates.")
    print("   Latent samples are already in the same order as the raw test split — no reindexing needed.")

In [ ]:
# ── t-SNE hyperparameters ─────────────────────────────────────────────────────
N_SUBSAMPLE_PER_CLIMATE = 2500
TSNE_PERPLEXITY         = 50
TSNE_N_ITER             = 1000

# All physical invariants — excludes the trailing anomaly features
TSNE_INVARIANTS = [
    "RH_surface", "RH850", "RH500",
    "lapse_850_500", "surface_850_temp_diff",
    "shear_u", "shear_v", "shear_mag",
]

# ── extract_test_latent_and_physical ─────────────────────────────────────────
# augmented_features_by_climate = [X_scaled | P_scaled] concatenated.
# The first n_input_variables * grid_points_per_patch columns are raw scaled
# inputs; the rest are physical features (n_physical_variables * grid_points_per_patch).
_n_raw_cols = n_input_variables * grid_points_per_patch


def extract_test_latent_and_physical(latent_key, n_per_climate, seed=42):
    """Return subsampled (Z, P, clim_list) for the test split of each climate.

    Z    : (n_total, latent_dim)         latent vectors
    P    : (n_total, n_physical_variables) physical features averaged over grid points
    clim : list of climate labels, one per sample
    """
    rng = np.random.default_rng(seed)
    Z_parts, P_parts, clim_parts = [], [], []

    for c in climate_order:
        # ── Latent vectors ────────────────────────────────────────────────────
        Z_all = latent_test_sets[latent_key]["latent_test_by_climate"][c]

        # ── Physical features from the augmented array ────────────────────────
        # For historical, augmented_features_by_climate still contains all
        # samples, so we restrict to the test split first.
        aug = augmented_features_by_climate[c]
        if c == "historical":
            aug = aug[ae_split_indices["historical"]["test"]]

        # Physical block: columns after the raw input features
        P_flat = aug[:, _n_raw_cols:]   # (N, n_physical_variables * grid_points_per_patch)

        # Average over grid points → (N, n_physical_variables)
        P_all = P_flat.reshape(len(P_flat), n_physical_variables, grid_points_per_patch).mean(axis=2)

        # ── Apply reindex if metadata order differs ───────────────────────────
        reindex = reindex_maps[latent_key][c]
        if reindex is not None:
            P_all = P_all[reindex]

        # ── Subsample ─────────────────────────────────────────────────────────
        n_avail = len(Z_all)
        n = min(n_per_climate, n_avail)
        idx = rng.choice(n_avail, size=n, replace=False)

        Z_parts.append(np.asarray(Z_all[idx], dtype=np.float32))
        P_parts.append(np.asarray(P_all[idx], dtype=np.float32))
        clim_parts.extend([c] * n)

    return np.vstack(Z_parts), np.vstack(P_parts), clim_parts


In [ ]:
print("Extracting test-split data for all latent setups…")
Z_cera,      P_cera,      clim_cera      = extract_test_latent_and_physical(
    "exp5_cera",             N_SUBSAMPLE_PER_CLIMATE)
Z_baseline,  P_baseline,  clim_baseline  = extract_test_latent_and_physical(
    "exp5_baseline_noalign",        N_SUBSAMPLE_PER_CLIMATE)
Z_cera_full, P_cera_full, clim_cera_full = extract_test_latent_and_physical(
    "exp5_cera_full_latent", N_SUBSAMPLE_PER_CLIMATE)
Z_climax,    P_climax,    clim_climax    = extract_test_latent_and_physical(
    "exp5_baseline_climax",  N_SUBSAMPLE_PER_CLIMATE)
Z_cera_exp3, P_cera_exp3, clim_cera_exp3 = extract_test_latent_and_physical(
    "exp3_AEall",            N_SUBSAMPLE_PER_CLIMATE)

# CERA aligned: first 48 dimensions of the exp5_cera latent (those subject to SWD
# alignment) — NOT to be confused with exp5_cera_full_latent, a separate run
# where the full latent is given to the predictor.
Z_cera_aligned = Z_cera[:, :48]

# Baseline (no-align): only the first 48 dims are actually given to its predictor
# (same split as CERA's 48 aligned / 16 free dims), so only those are used here too.
Z_baseline = Z_baseline[:, :48]

print(f"  CERA               : latent {Z_cera.shape},         physical {P_cera.shape}")
print(f"  CERA aligned       : latent {Z_cera_aligned.shape}")
print(f"  CERA full_latent   : latent {Z_cera_full.shape},         physical {P_cera_full.shape}")
print(f"  Baseline no Align  : latent {Z_baseline.shape}, physical {P_baseline.shape}")
print(f"  Baseline ClimaX    : latent {Z_climax.shape},   physical {P_climax.shape}")
print(f"  exp3 AEall     : latent {Z_cera_exp3.shape}, physical {P_cera_exp3.shape}")
print(f"  Climates           : {climate_order}")

In [ ]:
def _run_tsne(Z, label):
    print(f"Running t-SNE on {label} latent space  (n={len(Z)})…")
    return TSNE(
        n_components=2, perplexity=TSNE_PERPLEXITY, max_iter=TSNE_N_ITER,
        random_state=random_seed, n_jobs=-1,
    ).fit_transform(Z)


tsne_cera              = _run_tsne(Z_cera,              "CERA (64D)")
tsne_cera_aligned      = _run_tsne(Z_cera_aligned,      "CERA aligned (48D)")
tsne_cera_full         = _run_tsne(Z_cera_full,         "CERA full_latent (64D)")
tsne_baseline          = _run_tsne(Z_baseline,          "Baseline (no alignment, 48D)")
tsne_climax            = _run_tsne(Z_climax,            "Baseline ClimaX")
tsne_cera_exp3      = _run_tsne(Z_cera_exp3,      "CERA exp3 (64D)")

print(
    "Done.  Shapes: "
    f"CERA={tsne_cera.shape},  CERA_aligned={tsne_cera_aligned.shape},  "
    f"CERA_full_latent={tsne_cera_full.shape},  "
    f"Baseline={tsne_baseline.shape},  ClimaX={tsne_climax.shape},  exp3={tsne_cera_exp3.shape}"
)

In [ ]:
# ── 2-A  t-SNE coloured by climate ────────────────────────────────────────────
tsne_setups = [
    (tsne_cera_aligned, clim_cera,      "CERA — aligned 48D (exp5)"),
    (tsne_cera_full,    clim_cera_full, "CERA full_latent — 64D to predictor (exp5)"),
    (tsne_baseline,     clim_baseline,  "Baseline — no alignment, 48D (exp5)"),
    (tsne_climax,       clim_climax,    "Baseline — ClimaX (exp5)"),
    (tsne_cera_exp3,      clim_cera_exp3, "CERA — exp3 AEall 64D"),
]

ncols = 3
nrows = (len(tsne_setups) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 7 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).flatten()

for ax, (emb, clim_list, title) in zip(axes, tsne_setups):
    clim_arr = np.array(clim_list)
    for c in climate_order:
        mask = clim_arr == c
        ax.scatter(
            emb[mask, 0], emb[mask, 1],
            color=climate_colors.get(c, f"C{climate_order.index(c)}"),
            alpha=0.45, s=8, label=c, rasterized=True,
        )
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("t-SNE 1", fontsize=10)
    ax.set_ylabel("t-SNE 2", fontsize=10)
    ax.legend(markerscale=3, frameon=False, fontsize=9)
    ax.grid(alpha=0.2)

for ax in axes[len(tsne_setups):]:
    ax.set_visible(False)

fig.suptitle(
    "t-SNE of latent spaces — coloured by climate\n"
    "Aligned CERA should mix climates; Baseline / ClimaX / exp3 should keep them separated",
    fontsize=14,
)
plt.show()

In [ ]:
# ── 2-B  Latent t-SNE coloured by physical invariants ─────────────────────────
# One row block per setup. Spatial coherence of colour within a t-SNE cluster
# → the model encoded that invariant.

model_blocks = [
    (tsne_cera_aligned, P_cera,      "CERA aligned (48D)"),
    (tsne_cera_full,    P_cera_full, "CERA full_latent (64D)"),
    (tsne_baseline,     P_baseline,  "Baseline (no alignment, 48D)"),
    (tsne_climax,       P_climax,    "Baseline ClimaX"),
    (tsne_cera_exp3,      P_cera_exp3, "CERA exp3 (64D)")
]

n_inv  = len(TSNE_INVARIANTS)
ncols  = 4
nrows_per_model = (n_inv + ncols - 1) // ncols
nrows = nrows_per_model * len(model_blocks)

fig, axes = plt.subplots(nrows, ncols,
                         figsize=(ncols * 5, nrows * 4.5),
                         constrained_layout=True)

for row_offset, (emb, P_source, label) in enumerate(model_blocks):
    axes_block = axes[row_offset * nrows_per_model:(row_offset + 1) * nrows_per_model].flatten()

    for idx, inv_name in enumerate(TSNE_INVARIANTS):
        ax = axes_block[idx]
        pi        = physical_variable_names.index(inv_name)
        inv_vals  = P_source[:, pi]
        vmin, vmax = np.percentile(inv_vals, 2), np.percentile(inv_vals, 98)

        sc = ax.scatter(
            emb[:, 0], emb[:, 1],
            c=inv_vals, cmap="RdYlBu_r",
            alpha=0.5, s=7, vmin=vmin, vmax=vmax, rasterized=True,
        )
        plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(f"{label} — {inv_name}", fontsize=10, fontweight="bold")
        ax.set_xlabel("t-SNE 1", fontsize=9)
        ax.set_ylabel("t-SNE 2", fontsize=9)
        ax.grid(alpha=0.2)

    for idx in range(n_inv, len(axes_block)):
        axes_block[idx].set_visible(False)

fig.suptitle(
    "Latent space (t-SNE) — coloured by physical invariants, one row block per setup\n"
    "Colour coherence within clusters signals that the model encoded that physical quantity",
    fontsize=13, y=1.01,
)
plt.show()

In [ ]:
# ── 2-C  All setups — side-by-side per invariant ──────────────────────────────
# Shared colour scale across all panels for a direct comparison.

model_series = [
    (tsne_cera,         P_cera,      "CERA (64D)"),
    (tsne_cera_aligned, P_cera,      "CERA aligned (48D)"),
    (tsne_cera_full,    P_cera_full, "CERA full_latent (64D)"),
    (tsne_baseline,     P_baseline,  "Baseline (no alignment, 48D)"),
    (tsne_climax,       P_climax,    "Baseline ClimaX"),
    (tsne_cera_exp3,      P_cera_exp3, "CERA exp3 (64D)")
]

for inv_name in TSNE_INVARIANTS:
    pi = physical_variable_names.index(inv_name)
    inv_values_list = [P_source[:, pi] for _, P_source, _ in model_series]

    vmin = np.percentile(np.concatenate(inv_values_list), 2)
    vmax = np.percentile(np.concatenate(inv_values_list), 98)

    fig, axes = plt.subplots(1, len(model_series), figsize=(8 * len(model_series), 6), constrained_layout=True)

    for ax, (emb, P_source, title) in zip(axes, model_series):
        vals = P_source[:, pi]
        sc = ax.scatter(
            emb[:, 0], emb[:, 1],
            c=vals, cmap="RdYlBu_r",
            alpha=0.5, s=7, vmin=vmin, vmax=vmax, rasterized=True,
        )
        plt.colorbar(sc, ax=ax, label=inv_name)
        ax.set_title(f"{title} — {inv_name}", fontsize=12, fontweight="bold")
        ax.set_xlabel("t-SNE 1", fontsize=10)
        ax.set_ylabel("t-SNE 2", fontsize=10)
        ax.grid(alpha=0.2)

    fig.suptitle(
        f"All setups — latent space (t-SNE) coloured by {inv_name}\n"
        "Colour clusters aligned with geometric clusters → model encoded this physical feature",
        fontsize=13,
    )
    plt.show()

## Part 3 - Analysis of invariants in latent representations using predictions

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# ── Parameters ────────────────────────────────────────────────────────────────
N_LINEAR_PROBE         = 20000   # samples per climate  →  25 000 total
LINEAR_PROBE_TEST_FRAC = 0.20   # 80 % train / 20 % evaluation
RIDGE_ALPHA            = 1.0

PROBE_MODELS = ["CERA", "CERA_aligned", "CERA_full_latent", "Baseline", "ClimaX", "Exp3_AEall"]
PROBE_COLORS = {
    "CERA":             "#5B8DB8",
    "CERA_aligned":     "#8FBCBB",
    "CERA_full_latent": "#3B6E8F",
    "Baseline":         "#E07B39",
    "ClimaX":           "#9B59B6",
    "Exp3_AEall":       "#3498DB",
}

# Extract a larger subsample than t-SNE (different seed to get independent samples)
Z_cera_lp,      P_cera_lp,      _ = extract_test_latent_and_physical(
    "exp5_cera",             N_LINEAR_PROBE, seed=random_seed + 200)
Z_base_lp,      P_base_lp,      _ = extract_test_latent_and_physical(
    "exp5_baseline2",        N_LINEAR_PROBE, seed=random_seed + 200)
Z_cera_full_lp, P_cera_full_lp, _ = extract_test_latent_and_physical(
    "exp5_cera_full_latent", N_LINEAR_PROBE, seed=random_seed + 200)
Z_climax_lp,    P_climax_lp,    _ = extract_test_latent_and_physical(
    "exp5_baseline_climax",  N_LINEAR_PROBE, seed=random_seed + 200)
Z_cera_exp3_lp, P_cera_exp3_lp, _ = extract_test_latent_and_physical(
    "exp5_cera_exp3",        N_LINEAR_PROBE, seed=random_seed + 200)
# CERA aligned: first 48 dimensions
Z_cera_aligned_lp = Z_cera_lp[:, :48]

# Baseline (no-align): only the first 48 dims are actually given to its predictor
Z_base_lp = Z_base_lp[:, :48]

print(
    "Linear probing data — "
    f"CERA: {Z_cera_lp.shape}, CERA aligned: {Z_cera_aligned_lp.shape}, "
        f"CERA full_latent: {Z_cera_full_lp.shape}, Baseline: {Z_base_lp.shape}, "
        f"ClimaX: {Z_climax_lp.shape}, CERA exp3: {Z_cera_exp3_lp.shape}"
)

# ── Fit one Ridge per (model × invariant) ────────────────────────────────────
lp_rows = []

for model_name, Z, P in [
    ("CERA",             Z_cera_lp,         P_cera_lp),
    ("CERA_aligned",     Z_cera_aligned_lp, P_cera_lp),
    ("CERA_full_latent", Z_cera_full_lp,    P_cera_full_lp),
    ("Baseline",         Z_base_lp,         P_base_lp),
    ("ClimaX",           Z_climax_lp,       P_climax_lp),
    ("Exp3_AEall",       Z_cera_exp3_lp,    P_cera_exp3_lp),
]:
    Z_tr, Z_te, P_tr, P_te = train_test_split(
        Z, P, test_size=LINEAR_PROBE_TEST_FRAC, random_state=random_seed)

    for inv_name in TSNE_INVARIANTS:
        pi   = physical_variable_names.index(inv_name)
        y_tr = P_tr[:, pi]
        y_te = P_te[:, pi]

        ridge = Ridge(alpha=RIDGE_ALPHA)
        ridge.fit(Z_tr, y_tr)
        r2 = float(r2_score(y_te, ridge.predict(Z_te)))

        lp_rows.append({"model": model_name, "invariant": inv_name, "R2_linear": r2})
    print(f"  {model_name} done.")

linear_probe_df = pd.DataFrame(lp_rows)

# ── Summary table ─────────────────────────────────────────────────────────────
lp_pivot = (linear_probe_df
            .pivot(index="invariant", columns="model", values="R2_linear")
            .loc[TSNE_INVARIANTS, PROBE_MODELS])
lp_pivot.columns.name = None

lp_gain_cols = []
for model_name in PROBE_MODELS:
    if model_name == "Baseline":
        continue
    col = f"{model_name} − Baseline"
    lp_pivot[col] = lp_pivot[model_name] - lp_pivot["Baseline"]
    lp_gain_cols.append(col)

n_train = int((1 - LINEAR_PROBE_TEST_FRAC) * N_LINEAR_PROBE * len(climate_order))
n_test  = int(LINEAR_PROBE_TEST_FRAC       * N_LINEAR_PROBE * len(climate_order))

display(
    lp_pivot.style
    .format("{:.4f}")
    .background_gradient(subset=PROBE_MODELS, axis=None, cmap="YlGn")
    .background_gradient(subset=lp_gain_cols, axis=None, cmap="RdYlGn")
    .set_caption(
        f"Linear Probing R²  (Ridge α={RIDGE_ALPHA},  "
        f"{n_train} train / {n_test} test samples) — "
        "higher = invariant more linearly accessible from the latent"
    )
    .set_table_styles([{"selector": "caption",
                        "props": [("font-size", "13px"), ("font-weight", "bold")]}])
)

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 5), constrained_layout=True)
x        = np.arange(len(TSNE_INVARIANTS))
n_models = len(PROBE_MODELS)
width    = 0.8 / n_models

ax = axes[0]
for i, model_name in enumerate(PROBE_MODELS):
    ax.bar(x + (i - (n_models - 1) / 2) * width, lp_pivot[model_name], width,
           label=model_name, color=PROBE_COLORS[model_name], alpha=0.88)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(TSNE_INVARIANTS, rotation=45, ha="right")
ax.set_ylabel("R²")
ax.set_ylim(bottom=min(0, lp_pivot[PROBE_MODELS].values.min() - 0.05))
ax.set_title("Linear Probing R² per model"); ax.legend(frameon=False, fontsize=8); ax.grid(axis="y", alpha=0.3)

ax = axes[1]
n_gain  = len(lp_gain_cols)
width_g = 0.8 / n_gain
for i, col in enumerate(lp_gain_cols):
    model_name = col.split(" − ")[0]
    colors = [PROBE_COLORS[model_name] if v >= 0 else "#c0392b" for v in lp_pivot[col]]
    ax.bar(x + (i - (n_gain - 1) / 2) * width_g, lp_pivot[col], width_g,
           color=colors, alpha=0.88, label=col)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(TSNE_INVARIANTS, rotation=45, ha="right")
ax.set_ylabel("ΔR²")
ax.set_title("Gain over Baseline\n(positive = better encoding than Baseline)")
ax.legend(frameon=False, fontsize=7); ax.grid(axis="y", alpha=0.3)

fig.suptitle(
    "Linear Probing R² — Ridge regression from latent → physical invariant\n"
    f"({N_LINEAR_PROBE} samples per climate × {len(climate_order)} climates, "
    f"Ridge α={RIDGE_ALPHA})",
    fontsize=13,
)
plt.show()

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# ── Parameters ────────────────────────────────────────────────────────────────
N_KNN         = 10000   # samples per climate  (KNN scales as O(n_train × n_test))
KNN_TEST_FRAC = 0.20   # 80 % train / 20 % evaluation
KNN_K         = 10

PROBE_MODELS = ["CERA", "CERA_aligned", "CERA_full_latent", "Baseline", "ClimaX", "Exp3_AEall"]
PROBE_COLORS = {
    "CERA":             "#5B8DB8",
    "CERA_aligned":     "#8FBCBB",
    "CERA_full_latent": "#3B6E8F",
    "Baseline":         "#E07B39",
    "ClimaX":           "#9B59B6",
    "Exp3_AEall":       "#3498DB",
}

# Extract subsample (different seed from linear probing)
Z_cera_knn,      P_cera_knn,      _ = extract_test_latent_and_physical(
    "exp5_cera",             N_KNN, seed=random_seed + 300)
Z_base_knn,      P_base_knn,      _ = extract_test_latent_and_physical(
    "exp5_baseline2",        N_KNN, seed=random_seed + 300)
Z_cera_full_knn, P_cera_full_knn, _ = extract_test_latent_and_physical(
    "exp5_cera_full_latent", N_KNN, seed=random_seed + 300)
Z_climax_knn,    P_climax_knn,    _ = extract_test_latent_and_physical(
    "exp5_baseline_climax",  N_KNN, seed=random_seed + 300)
Z_cera_exp3_knn, P_cera_exp3_knn, _ = extract_test_latent_and_physical(
    "exp5_cera_exp3",        N_KNN, seed=random_seed + 300)

# CERA aligned: first 48 dimensions
Z_cera_aligned_knn = Z_cera_knn[:, :48]

# Baseline (no-align): only the first 48 dims are actually given to its predictor
Z_base_knn = Z_base_knn[:, :48]

print(
    "k-NN probing data — "
    f"CERA: {Z_cera_knn.shape}, CERA aligned: {Z_cera_aligned_knn.shape}, "
    f"CERA full_latent: {Z_cera_full_knn.shape}, Baseline: {Z_base_knn.shape}, "
    f"ClimaX: {Z_climax_knn.shape}, CERA exp3: {Z_cera_exp3_knn.shape}"
)

# ── Fit one KNN regressor per (model × invariant) ────────────────────────────
knn_rows = []

for model_name, Z, P in [
    ("CERA",             Z_cera_knn,         P_cera_knn),
    ("CERA_aligned",     Z_cera_aligned_knn, P_cera_knn),
    ("CERA_full_latent", Z_cera_full_knn,    P_cera_full_knn),
    ("Baseline",         Z_base_knn,         P_base_knn),
    ("ClimaX",           Z_climax_knn,       P_climax_knn),
    ("Exp3_AEall",       Z_cera_exp3_knn,    P_cera_exp3_knn),
]:
    Z_tr, Z_te, P_tr, P_te = train_test_split(
        Z, P, test_size=KNN_TEST_FRAC, random_state=random_seed)

    for inv_name in TSNE_INVARIANTS:
        pi   = physical_variable_names.index(inv_name)
        y_tr = P_tr[:, pi]
        y_te = P_te[:, pi]

        knn = KNeighborsRegressor(n_neighbors=KNN_K, n_jobs=-1)
        knn.fit(Z_tr, y_tr)
        r2 = float(r2_score(y_te, knn.predict(Z_te)))

        knn_rows.append({"model": model_name, "invariant": inv_name, "R2_knn": r2})
    print(f"  {model_name} done.")

knn_probe_df = pd.DataFrame(knn_rows)

# ── Summary table ─────────────────────────────────────────────────────────────
knn_pivot = (knn_probe_df
             .pivot(index="invariant", columns="model", values="R2_knn")
             .loc[TSNE_INVARIANTS, PROBE_MODELS])
knn_pivot.columns.name = None

knn_gain_cols = []
for model_name in PROBE_MODELS:
    if model_name == "Baseline":
        continue
    col = f"{model_name} − Baseline"
    knn_pivot[col] = knn_pivot[model_name] - knn_pivot["Baseline"]
    knn_gain_cols.append(col)

n_train_k = int((1 - KNN_TEST_FRAC) * N_KNN * len(climate_order))
n_test_k  = int(KNN_TEST_FRAC       * N_KNN * len(climate_order))

display(
    knn_pivot.style
    .format("{:.4f}")
    .background_gradient(subset=PROBE_MODELS, axis=None, cmap="YlGn")
    .background_gradient(subset=knn_gain_cols, axis=None, cmap="RdYlGn")
    .set_caption(
        f"k-NN Regression R²  (k={KNN_K},  "
        f"{n_train_k} train / {n_test_k} test samples) — "
        "higher = locally similar latent points share similar invariant values"
    )
    .set_table_styles([{"selector": "caption",
                        "props": [("font-size", "13px"), ("font-weight", "bold")]}])
)

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 5), constrained_layout=True)
x        = np.arange(len(TSNE_INVARIANTS))
n_models = len(PROBE_MODELS)
width    = 0.8 / n_models

ax = axes[0]
for i, model_name in enumerate(PROBE_MODELS):
    ax.bar(x + (i - (n_models - 1) / 2) * width, knn_pivot[model_name], width,
           label=model_name, color=PROBE_COLORS[model_name], alpha=0.88)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(TSNE_INVARIANTS, rotation=45, ha="right")
ax.set_ylabel("R²")
ax.set_ylim(bottom=min(0, knn_pivot[PROBE_MODELS].values.min() - 0.05))
ax.set_title(f"k-NN R² per model  (k={KNN_K})"); ax.legend(frameon=False, fontsize=8); ax.grid(axis="y", alpha=0.3)

ax = axes[1]
n_gain  = len(knn_gain_cols)
width_g = 0.8 / n_gain
for i, col in enumerate(knn_gain_cols):
    model_name = col.split(" − ")[0]
    colors = [PROBE_COLORS[model_name] if v >= 0 else "#c0392b" for v in knn_pivot[col]]
    ax.bar(x + (i - (n_gain - 1) / 2) * width_g, knn_pivot[col], width_g,
           color=colors, alpha=0.88, label=col)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(TSNE_INVARIANTS, rotation=45, ha="right")
ax.set_ylabel("ΔR²")
ax.set_title("Gain over Baseline\n(positive = better encoding than Baseline)")
ax.legend(frameon=False, fontsize=7); ax.grid(axis="y", alpha=0.3)

fig.suptitle(
    f"k-NN Regression R² (k={KNN_K}) — from latent → physical invariant\n"
    f"({N_KNN} samples per climate × {len(climate_order)} climates)",
    fontsize=13,
)
plt.show()

# ── Combined summary : Linear vs k-NN ────────────────────────────────────────
combined_cols = {}
for model_name in PROBE_MODELS:
    combined_cols[f"Linear {model_name}"] = lp_pivot[model_name]
for model_name in PROBE_MODELS:
    if model_name == "Baseline":
        continue
    combined_cols[f"Linear {model_name}−Base"] = lp_pivot[f"{model_name} − Baseline"]
for model_name in PROBE_MODELS:
    combined_cols[f"KNN-{KNN_K} {model_name}"] = knn_pivot[model_name]
for model_name in PROBE_MODELS:
    if model_name == "Baseline":
        continue
    combined_cols[f"KNN-{KNN_K} {model_name}−Base"] = knn_pivot[f"{model_name} − Baseline"]

combined = pd.DataFrame(combined_cols).loc[TSNE_INVARIANTS]

gain_gradient_cols = [c for c in combined.columns if "−Base" in c]

display(
    combined.style
    .format("{:.4f}")
    .background_gradient(subset=gain_gradient_cols, axis=None, cmap="RdYlGn")
    .set_caption(
        "Combined summary — Linear Probing R² vs k-NN R²\n"
        "Model−Base > 0 → better encoding than Baseline"
    )
    .set_table_styles([{"selector": "caption",
                        "props": [("font-size", "13px"), ("font-weight", "bold")]}])
)

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# ── Parameters ────────────────────────────────────────────────────────────────
N_KNN         = 10000   # samples per climate  (KNN scales as O(n_train × n_test))
KNN_TEST_FRAC = 0.20   # 80 % train / 20 % evaluation
KNN_K         = 10

PROBE_MODELS = ["CERA_aligned", "CERA_full_latent", "Baseline", "ClimaX", "Exp3_AEall"]
PROBE_COLORS = {
    "CERA_aligned":     "#275fc4",
    "CERA_full_latent": "#10337d",
    "Baseline":         "#6b92c0",
    "ClimaX":           "#c57fca",
    "Exp3_AEall":       "#3498DB",
}

# Extract subsample (different seed from linear probing)
Z_cera_knn,      P_cera_knn,      _ = extract_test_latent_and_physical(
    "exp5_cera",             N_KNN, seed=random_seed + 300)
Z_base_knn,      P_base_knn,      _ = extract_test_latent_and_physical(
    "exp5_baseline2",        N_KNN, seed=random_seed + 300)
Z_cera_full_knn, P_cera_full_knn, _ = extract_test_latent_and_physical(
    "exp5_cera_full_latent", N_KNN, seed=random_seed + 300)
Z_climax_knn,    P_climax_knn,    _ = extract_test_latent_and_physical(
    "exp5_baseline_climax",  N_KNN, seed=random_seed + 300)
Z_cera_exp3_knn, P_cera_exp3_knn, _ = extract_test_latent_and_physical(
    "exp5_cera_exp3",        N_KNN, seed=random_seed + 300)

# CERA aligned: first 48 dimensions
Z_cera_aligned_knn = Z_cera_knn[:, :48]

# Baseline (no-align): only the first 48 dims are actually given to its predictor
Z_base_knn = Z_base_knn[:, :48]

print(
    "k-NN probing data — "
    f"CERA aligned: {Z_cera_aligned_knn.shape}, "
    f"CERA full_latent: {Z_cera_full_knn.shape}, Baseline: {Z_base_knn.shape}, "
    f"ClimaX: {Z_climax_knn.shape}, CERA exp3: {Z_cera_exp3_knn.shape}"
)

# ── Fit one KNN regressor per (model × invariant) ────────────────────────────
knn_rows = []

for model_name, Z, P in [
    ("CERA_aligned",     Z_cera_aligned_knn,      P_cera_knn),
    ("CERA_full_latent", Z_cera_full_knn,         P_cera_full_knn),
    ("Baseline",         Z_base_knn,              P_base_knn),
    ("ClimaX",           Z_climax_knn,            P_climax_knn),
    ("Exp3_AEall",       Z_cera_exp3_knn,         P_cera_exp3_knn),
]:
    Z_tr, Z_te, P_tr, P_te = train_test_split(
        Z, P, test_size=KNN_TEST_FRAC, random_state=random_seed)

    for inv_name in TSNE_INVARIANTS:
        pi   = physical_variable_names.index(inv_name)
        y_tr = P_tr[:, pi]
        y_te = P_te[:, pi]

        knn = KNeighborsRegressor(n_neighbors=KNN_K, n_jobs=-1)
        knn.fit(Z_tr, y_tr)
        r2 = float(r2_score(y_te, knn.predict(Z_te)))

        knn_rows.append({"model": model_name, "invariant": inv_name, "R2_knn": r2})
    print(f"  {model_name} done.")

knn_probe_df = pd.DataFrame(knn_rows)

# ── Summary table ─────────────────────────────────────────────────────────────
knn_pivot = (knn_probe_df
             .pivot(index="invariant", columns="model", values="R2_knn")
             .loc[TSNE_INVARIANTS, PROBE_MODELS])
knn_pivot.columns.name = None

knn_gain_cols = []
for model_name in PROBE_MODELS:
    if model_name == "Baseline":
        continue
    col = f"{model_name} − Baseline"
    knn_pivot[col] = knn_pivot[model_name] - knn_pivot["Baseline"]
    knn_gain_cols.append(col)

n_train_k = int((1 - KNN_TEST_FRAC) * N_KNN * len(climate_order))
n_test_k  = int(KNN_TEST_FRAC       * N_KNN * len(climate_order))

display(
    knn_pivot.style
    .format("{:.4f}")
    .background_gradient(subset=PROBE_MODELS, axis=None, cmap="YlGn")
    .background_gradient(subset=knn_gain_cols, axis=None, cmap="RdYlGn")
    .set_caption("k-NN Regression R² from latent")
    .set_table_styles([{"selector": "caption",
                        "props": [("font-size", "13px"), ("font-weight", "bold")]}])
)

# ── Bar chart (kNN R² per model only) ────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(9, 5.5), constrained_layout=True)
x        = np.arange(len(TSNE_INVARIANTS))
n_models = len(PROBE_MODELS)
width    = 0.8 / n_models

for i, model_name in enumerate(PROBE_MODELS):
    ax.bar(x + (i - (n_models - 1) / 2) * width, knn_pivot[model_name], width,
           label=model_name, color=PROBE_COLORS[model_name], alpha=0.88)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(TSNE_INVARIANTS, rotation=45, ha="right")
ax.set_ylabel("R²")
ax.set_ylim(bottom=min(0.2, knn_pivot[PROBE_MODELS].values.min() - 0.05))
ax.legend(frameon=False, fontsize=9, loc="upper center",
          bbox_to_anchor=(0.5, -0.22), ncol=n_models)
ax.grid(axis="y", alpha=0.3)

fig.suptitle(
    "k-NN Regression R² from latent",
    fontsize=13,
)
plt.show()


## Part 4 - Cluster analysis of invariants

In [ ]:
# ── 4-0  HDBSCAN clustering of each latent setup ──────────────────────────────
# For each setup: take up to MAX_POINTS_PER_CLIMATE_FOR_FIT points per climate
# (all of them if fewer are available, otherwise a random subsample),
# concatenate across climates, and fit HDBSCAN on that sample — the number of
# clusters is found automatically, independently per setup. If a climate had
# more points available than the cap, the leftover points (left out of the
# fit) are assigned a cluster using HDBSCAN's predictive extension
# (hdbscan.approximate_predict). Points labelled as noise (-1) by either the
# fit or the prediction are kept aside — they are excluded from the cluster
# analyses in the cells below.
#
# HDBSCAN_MIN_CLUSTER_SIZE / HDBSCAN_MIN_SAMPLES were raised above their
# library defaults (5 / 5): with the default values, a handful of near-
# duplicate points could form their own "cluster" (e.g. a 5-point cluster was
# found for CERA_aligned) and noise was distributed quite unevenly across
# climates. min_cluster_size filters out these small spurious clusters;
# min_samples makes the density estimate underlying the whole clustering more
# conservative (more robust, but also somewhat more noise overall).

import hdbscan

MAX_POINTS_PER_CLIMATE_FOR_FIT = 30000
HDBSCAN_MIN_CLUSTER_SIZE       = 200   # ~2% of a setup's ~10k points; filters out tiny spurious clusters
HDBSCAN_MIN_SAMPLES            = 50    # more conservative / robust density estimate than the default (5)

CLUSTER_MODELS = ["CERA", "CERA_aligned", "CERA_full_latent", "Baseline", "ClimaX", "Exp3_AEall"]
CLUSTER_LATENT_KEYS = {
    "CERA":             "exp5_cera",
    "CERA_aligned":     "exp5_cera",              # sliced to the first 48 aligned dims below
    "CERA_full_latent": "exp5_cera_full_latent",
    "Baseline":         "exp5_baseline2",
    "ClimaX":           "exp5_baseline_climax",
    "Exp3_AEall":       "exp5_cera_exp3"
}

# Extract every available test-split point once per underlying latent_key
# (CERA and CERA_aligned share the same exp5_cera extraction).
_all_Z, _all_P, _all_clim = {}, {}, {}
for latent_key in set(CLUSTER_LATENT_KEYS.values()):
    Z, P, clim = extract_test_latent_and_physical(latent_key, n_per_climate=10**9)
    _all_Z[latent_key], _all_P[latent_key], _all_clim[latent_key] = Z, P, clim

cluster_data = {}   # {model_name: {"Z", "P", "clim", "labels", "clusterer"}}
_cluster_rng = np.random.default_rng(random_seed)

print(f"Clustering each setup with HDBSCAN (min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE}, "
      f"min_samples={HDBSCAN_MIN_SAMPLES}, fit cap = {MAX_POINTS_PER_CLIMATE_FOR_FIT} pts/climate)…")
for model_name in CLUSTER_MODELS:
    latent_key = CLUSTER_LATENT_KEYS[model_name]
    Z = _all_Z[latent_key]
    if model_name in ("CERA_aligned", "Baseline", "Exp3_AEall"):
        Z = Z[:, :48]
    P    = _all_P[latent_key]
    clim = np.array(_all_clim[latent_key])

    # ── Build the fit sample: up to MAX_POINTS_PER_CLIMATE_FOR_FIT pts/climate ──
    fit_idx_parts, leftover_idx_parts = [], []
    for c in climate_order:
        c_idx = np.where(clim == c)[0]
        if len(c_idx) > MAX_POINTS_PER_CLIMATE_FOR_FIT:
            chosen = _cluster_rng.choice(c_idx, size=MAX_POINTS_PER_CLIMATE_FOR_FIT, replace=False)
            fit_idx_parts.append(chosen)
            leftover_idx_parts.append(np.setdiff1d(c_idx, chosen, assume_unique=True))
        else:
            fit_idx_parts.append(c_idx)

    fit_idx      = np.concatenate(fit_idx_parts)
    leftover_idx = np.concatenate(leftover_idx_parts) if leftover_idx_parts else np.array([], dtype=int)

    clusterer  = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        prediction_data=True,
    )
    fit_labels = clusterer.fit_predict(Z[fit_idx])

    # ── Predictive completion for points left out of the fit sample ─────────
    if len(leftover_idx) > 0:
        leftover_labels, _ = hdbscan.approximate_predict(clusterer, Z[leftover_idx])
    else:
        leftover_labels = np.array([], dtype=fit_labels.dtype)

    labels                = np.empty(len(Z), dtype=fit_labels.dtype)
    labels[fit_idx]        = fit_labels
    labels[leftover_idx]   = leftover_labels

    n_noise    = int((labels == -1).sum())
    n_clusters = len(set(labels.tolist())) - (1 if -1 in labels else 0)

    cluster_data[model_name] = {"Z": Z, "P": P, "clim": clim, "labels": labels, "clusterer": clusterer}
    print(f"  {model_name:17s}: n={len(Z):6d}, latent_dim={Z.shape[1]:3d}, "
          f"clusters found={n_clusters:2d}, noise={n_noise:6d} ({100 * n_noise / len(Z):.1f}%), "
          f"fit sample={len(fit_idx):6d}, predicted={len(leftover_idx):6d}")


In [ ]:
# ── 4-1  Climate composition of each cluster ──────────────────────────────────
# HDBSCAN noise points (-1) are excluded from this analysis. For each setup:
# proportion of each climate within each cluster (bars) vs. the overall
# proportion of that climate among the non-noise points of that setup (dashed
# reference lines). Bars aligned with their matching dashed line → clusters
# mix climates in roughly the same proportions as the whole (non-noise) dataset.

ncols = 3
nrows = (len(CLUSTER_MODELS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5.5 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).flatten()

n_clim    = len(climate_order)
bar_width = 0.8 / n_clim

for ax, model_name in zip(axes, CLUSTER_MODELS):
    data   = cluster_data[model_name]
    labels = data["labels"]
    clim   = data["clim"]

    mask = labels != -1   # exclude HDBSCAN noise from the cluster analysis
    df = pd.DataFrame({"cluster": labels[mask], "climate": clim[mask]})
    counts = (df.groupby(["cluster", "climate"]).size()
                .unstack(fill_value=0)
                .reindex(columns=climate_order, fill_value=0)
                .sort_index())
    proportions = counts.div(counts.sum(axis=1), axis=0)
    global_proportions = df["climate"].value_counts(normalize=True).reindex(climate_order)

    x = np.arange(len(proportions.index))
    for i, c in enumerate(climate_order):
        color = climate_colors.get(c, f"C{i}")
        ax.bar(x + (i - (n_clim - 1) / 2) * bar_width, proportions[c], bar_width,
               color=color, alpha=0.9, label=c)
        ax.hlines(global_proportions[c], x.min() - 0.5, x.max() + 0.5,
                  color=color, linestyle="--", linewidth=1.2, alpha=0.8)

    n_noise = int((~mask).sum())
    ax.set_xticks(x)
    ax.set_xticklabels([f"C{k}" for k in proportions.index])
    ax.set_ylim(0, 1)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Proportion of climate within cluster")
    ax.set_title(f"{model_name}  ({len(proportions.index)} clusters, {n_noise} noise pts excluded)",
                 fontsize=10.5, fontweight="bold")
    ax.grid(axis="y", alpha=0.2)

for ax in axes[len(CLUSTER_MODELS):]:
    ax.set_visible(False)

axes[0].legend(frameon=False, fontsize=8, ncol=2, loc="upper left")
fig.suptitle(
    "Climate composition of each cluster (bars) vs. overall climate proportion among non-noise points (dashed lines)\n"
    "Bars aligned with their matching dashed line → clusters mix climates in roughly the same proportions as the whole dataset  (HDBSCAN noise excluded)",
    fontsize=13,
)
plt.show()


In [ ]:
# ── 4-2  Ridgeline (joyplot) of an invariant's distribution per cluster ───────
# Choose one invariant and one setup below. Each ridge is one cluster's KDE of
# that invariant, ranked by mean (bottom = lowest mean, top = highest), and
# coloured by where that mean sits on the invariant's value range using a
# continuous blue → white → red diverging colormap (low = blue, high = red).
# HDBSCAN noise points (-1) are excluded.

from scipy.stats import gaussian_kde

RIDGE_INVARIANT = "RH850"          # choose one of TSNE_INVARIANTS
RIDGE_MODEL     = "CERA_aligned"   # choose one of CLUSTER_MODELS


def plot_invariant_ridgeline(model_name, inv_name, ax=None):
    data   = cluster_data[model_name]
    labels = data["labels"]
    pi     = physical_variable_names.index(inv_name)

    mask   = labels != -1   # exclude HDBSCAN noise from the cluster analysis
    values = data["P"][mask, pi]
    labels = labels[mask]

    cluster_ids   = np.unique(labels)
    cluster_means = np.array([values[labels == cid].mean() for cid in cluster_ids])
    order         = np.argsort(cluster_means)
    cluster_ids   = cluster_ids[order]
    cluster_means = cluster_means[order]

    vmin, vmax = np.percentile(values, 1), np.percentile(values, 99)
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap("bwr")
    x_grid = np.linspace(vmin, vmax, 500)

    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 0.9 * len(cluster_ids) + 2))

    overlap = 0.7   # vertical spacing between ridges (in normalized density units)
    for i, (cid, mean_val) in enumerate(zip(cluster_ids, cluster_means)):
        vals_c  = values[labels == cid]
        density = gaussian_kde(vals_c)(x_grid)
        density = density / density.max()

        base_y = i * overlap
        color  = cmap(norm(mean_val))

        ax.fill_between(x_grid, base_y, base_y + density, color=color, alpha=0.85, zorder=-i)
        ax.plot(x_grid, base_y + density, color="black", linewidth=0.6, zorder=-i)
        ax.axhline(base_y, color="gray", linewidth=0.4, zorder=-i - 1)
        ax.text(vmin, base_y + 0.05, f"Cluster {cid}  (n={len(vals_c)})", fontsize=8, va="bottom")

    ax.set_yticks([])
    ax.set_xlim(vmin, vmax)
    ax.set_xlabel(inv_name)
    ax.set_title(
        f"{model_name} — {inv_name} distribution per cluster\n"
        "(ranked by mean, colour = position on the invariant axis; HDBSCAN noise excluded)",
        fontsize=12, fontweight="bold",
    )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label=inv_name, fraction=0.03, pad=0.02)

    return ax


plot_invariant_ridgeline(RIDGE_MODEL, RIDGE_INVARIANT)
plt.show()


In [ ]:
# ── 4-3  Aggregated view: intra-cluster vs. global variance ratio ─────────────
# For each (setup × invariant): mean intra-cluster variance (averaged over that
# setup's clusters) divided by the global variance of that invariant. Low
# (green) → clusters are compact / well separated along this invariant. High
# (red, ≈ 1) → clustering does not isolate this invariant at all. HDBSCAN noise
# points (-1) are excluded from both the intra-cluster and the global variance.

variance_ratio_rows = []

for model_name in CLUSTER_MODELS:
    data   = cluster_data[model_name]
    labels = data["labels"]
    P      = data["P"]

    mask      = labels != -1   # exclude HDBSCAN noise
    labels_nz = labels[mask]
    P_nz      = P[mask]

    for inv_name in TSNE_INVARIANTS:
        pi     = physical_variable_names.index(inv_name)
        values = P_nz[:, pi]

        global_var     = values.var()
        intra_vars     = [values[labels_nz == cid].var() for cid in np.unique(labels_nz)]
        mean_intra_var = np.mean(intra_vars)

        variance_ratio_rows.append({
            "model": model_name,
            "invariant": inv_name,
            "variance_ratio": mean_intra_var / global_var if global_var > 0 else np.nan,
        })

variance_ratio_df = pd.DataFrame(variance_ratio_rows)
variance_ratio_pivot = (variance_ratio_df
                        .pivot(index="invariant", columns="model", values="variance_ratio")
                        .loc[TSNE_INVARIANTS, CLUSTER_MODELS])

fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
im = ax.imshow(variance_ratio_pivot.values, cmap="RdYlGn_r", vmin=0, vmax=1, aspect="auto")

ax.set_xticks(np.arange(len(CLUSTER_MODELS)))
ax.set_xticklabels(CLUSTER_MODELS, rotation=45, ha="right")
ax.set_yticks(np.arange(len(TSNE_INVARIANTS)))
ax.set_yticklabels(TSNE_INVARIANTS)

for i in range(variance_ratio_pivot.shape[0]):
    for j in range(variance_ratio_pivot.shape[1]):
        ax.text(j, i, f"{variance_ratio_pivot.values[i, j]:.2f}",
                ha="center", va="center", color="black", fontsize=9)

plt.colorbar(im, ax=ax, label="Mean intra-cluster variance / global variance")
ax.set_title(
    "Intra-cluster vs. global variance ratio  (HDBSCAN, noise excluded)\n"
    "Low (green) = clusters strongly separate on this invariant  |  High (red, ≈1) = clustering doesn't isolate it",
    fontsize=12,
)
plt.show()


In [ ]:
# ── 4-4  Ridgeline per cluster, split by climate ──────────────────────────────
# Same idea as 4-2, but instead of a single distribution per cluster, each
# cluster shows one distribution per climate (coloured with climate_colors),
# so we can check whether an invariant's distribution within a cluster depends
# on climate. Uses the same RIDGE_MODEL / RIDGE_INVARIANT chosen in cell 4-2.
# HDBSCAN noise points (-1) are excluded.

def plot_invariant_ridgeline_by_climate(model_name, inv_name, ax=None):
    data   = cluster_data[model_name]
    labels = data["labels"]
    clim   = data["clim"]
    pi     = physical_variable_names.index(inv_name)

    mask   = labels != -1   # exclude HDBSCAN noise
    values = data["P"][mask, pi]
    labels = labels[mask]
    clim   = clim[mask]

    cluster_ids   = np.unique(labels)
    cluster_means = np.array([values[labels == cid].mean() for cid in cluster_ids])
    cluster_ids   = cluster_ids[np.argsort(cluster_means)]

    vmin, vmax = np.percentile(values, 1), np.percentile(values, 99)
    x_grid = np.linspace(vmin, vmax, 500)

    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 1.4 * len(cluster_ids) + 2))

    overlap = 1.0   # more vertical room than 4-2 since several climate curves share a ridge
    for i, cid in enumerate(cluster_ids):
        base_y       = i * overlap
        cluster_mask = labels == cid
        n_cluster    = int(cluster_mask.sum())

        for c in climate_order:
            vals_c = values[cluster_mask & (clim == c)]
            if len(vals_c) < 2:
                continue
            density = gaussian_kde(vals_c)(x_grid)
            density = density / density.max() * 0.9

            color = climate_colors.get(c, None)
            ax.plot(x_grid, base_y + density, color=color, linewidth=1.3, label=c)
            ax.fill_between(x_grid, base_y, base_y + density, color=color, alpha=0.15)

        ax.axhline(base_y, color="gray", linewidth=0.4, zorder=-i - 1)
        ax.text(vmin, base_y + 0.95, f"Cluster {cid}  (n={n_cluster})", fontsize=8, va="bottom")

    ax.set_yticks([])
    ax.set_xlim(vmin, vmax)
    ax.set_xlabel(inv_name)
    ax.set_title(
        f"{model_name} — {inv_name} distribution per cluster, split by climate\n"
        "(clusters ranked by overall mean; HDBSCAN noise excluded)",
        fontsize=12, fontweight="bold",
    )

    handles, labels_leg = ax.get_legend_handles_labels()
    by_label = dict(zip(labels_leg, handles))   # de-duplicate repeated climate labels across clusters
    ax.legend(by_label.values(), by_label.keys(), frameon=False, fontsize=8, loc="upper right")

    return ax


plot_invariant_ridgeline_by_climate(RIDGE_MODEL, RIDGE_INVARIANT)
plt.show()


## Part 4 bis - Cluster analysis of invariants using KMeans

HDBSCAN is unable to find any clusters (0 clusters and only noise for reasonable hyperparameters, and when we try to force the appearance of clusters by using small clusters, we get 2—with the second one having about 30 points), so this means that our latent space for CERA is indeed structured as a single block; we can therefore perform the LASSO part without clustering, but first we’ll verify what we’ve just stated by forcing clustering with K-Means to see how it behaves. --> Note that the numerical results I provided were for only 10,000 samples.

In [ ]:
# ── 4bis-0  K-Means clustering of each latent setup ───────────────────────────
# Same 6 setups as Part 4, but using K-Means instead of HDBSCAN: a fixed
# number of clusters (N_CLUSTERS) shared across all setups, fit on ALL
# available test-split points per setup (no subsampling). Unlike HDBSCAN,
# K-Means always assigns every point to a cluster — there is no "noise"
# category here, so the `labels != -1` filters in the cells below are simply
# no-ops in this part (K-Means never produces a -1 label).

from sklearn.cluster import KMeans

N_CLUSTERS = 4   # same K for every setup, to allow direct comparison

CLUSTER_MODELS = ["CERA", "CERA_aligned", "CERA_full_latent", "Baseline", "ClimaX", "Exp3_AEall"]
CLUSTER_LATENT_KEYS = {
    "CERA":             "exp5_cera",
    "CERA_aligned":     "exp5_cera",              # sliced to the first 48 aligned dims below
    "CERA_full_latent": "exp5_cera_full_latent",
    "Baseline":         "exp5_baseline2",
    "ClimaX":           "exp5_baseline_climax",
    "Exp3_AEall":       "exp5_cera_exp3"
}

# Extract every available test-split point once per underlying latent_key
# (CERA and CERA_aligned share the same exp5_cera extraction).
_all_Z, _all_P, _all_clim = {}, {}, {}
for latent_key in set(CLUSTER_LATENT_KEYS.values()):
    Z, P, clim = extract_test_latent_and_physical(latent_key, n_per_climate=10**9)
    _all_Z[latent_key], _all_P[latent_key], _all_clim[latent_key] = Z, P, clim

cluster_data = {}   # {model_name: {"Z", "P", "clim", "labels", "clusterer"}}

print(f"Clustering each setup with K-Means (K={N_CLUSTERS})…")
for model_name in CLUSTER_MODELS:
    latent_key = CLUSTER_LATENT_KEYS[model_name]
    Z = _all_Z[latent_key]
    if model_name in ("CERA_aligned", "Baseline", "Exp3_AEall"):
        Z = Z[:, :48]
    P    = _all_P[latent_key]
    clim = np.array(_all_clim[latent_key])

    clusterer = KMeans(n_clusters=N_CLUSTERS, random_state=random_seed, n_init=10)
    labels = clusterer.fit_predict(Z)

    cluster_data[model_name] = {"Z": Z, "P": P, "clim": clim, "labels": labels, "clusterer": clusterer}
    print(f"  {model_name:17s}: n={len(Z):6d}, latent_dim={Z.shape[1]:3d}, "
          f"cluster sizes={np.bincount(labels)}")


In [ ]:
# ── 4-1  Climate composition of each cluster ──────────────────────────────────
# HDBSCAN noise points (-1) are excluded from this analysis. For each setup:
# proportion of each climate within each cluster (bars) vs. the overall
# proportion of that climate among the non-noise points of that setup (dashed
# reference lines). Bars aligned with their matching dashed line → clusters
# mix climates in roughly the same proportions as the whole (non-noise) dataset.

ncols = 3
nrows = (len(CLUSTER_MODELS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5.5 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).flatten()

n_clim    = len(climate_order)
bar_width = 0.8 / n_clim

for ax, model_name in zip(axes, CLUSTER_MODELS):
    data   = cluster_data[model_name]
    labels = data["labels"]
    clim   = data["clim"]

    mask = labels != -1   # exclude HDBSCAN noise from the cluster analysis
    df = pd.DataFrame({"cluster": labels[mask], "climate": clim[mask]})
    counts = (df.groupby(["cluster", "climate"]).size()
                .unstack(fill_value=0)
                .reindex(columns=climate_order, fill_value=0)
                .sort_index())
    proportions = counts.div(counts.sum(axis=1), axis=0)
    global_proportions = df["climate"].value_counts(normalize=True).reindex(climate_order)

    x = np.arange(len(proportions.index))
    for i, c in enumerate(climate_order):
        color = climate_colors.get(c, f"C{i}")
        ax.bar(x + (i - (n_clim - 1) / 2) * bar_width, proportions[c], bar_width,
               color=color, alpha=0.9, label=c)
        ax.hlines(global_proportions[c], x.min() - 0.5, x.max() + 0.5,
                  color=color, linestyle="--", linewidth=1.2, alpha=0.8)

    n_noise = int((~mask).sum())
    ax.set_xticks(x)
    ax.set_xticklabels([f"C{k}" for k in proportions.index])
    ax.set_ylim(0, 1)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Proportion of climate within cluster")
    ax.set_title(f"{model_name}  ({len(proportions.index)} clusters, {n_noise} noise pts excluded)",
                 fontsize=10.5, fontweight="bold")
    ax.grid(axis="y", alpha=0.2)

for ax in axes[len(CLUSTER_MODELS):]:
    ax.set_visible(False)

axes[0].legend(frameon=False, fontsize=8, ncol=2, loc="upper left")
fig.suptitle(
    "Climate composition of each cluster (bars) vs. overall climate proportion among non-noise points (dashed lines)\n"
    "Bars aligned with their matching dashed line → clusters mix climates in roughly the same proportions as the whole dataset  (HDBSCAN noise excluded)",
    fontsize=13,
)
plt.show()


In [ ]:
# ── 4-2  Ridgeline (joyplot) of an invariant's distribution per cluster ───────
# Choose one invariant and one setup below. Each ridge is one cluster's KDE of
# that invariant, ranked by mean (bottom = lowest mean, top = highest), and
# coloured by where that mean sits on the invariant's value range using a
# continuous blue → white → red diverging colormap (low = blue, high = red).
# HDBSCAN noise points (-1) are excluded.

from scipy.stats import gaussian_kde

RIDGE_INVARIANT = "RH850"          # choose one of TSNE_INVARIANTS
RIDGE_MODEL     = "CERA"   # choose one of CLUSTER_MODELS


def plot_invariant_ridgeline(model_name, inv_name, ax=None):
    data   = cluster_data[model_name]
    labels = data["labels"]
    pi     = physical_variable_names.index(inv_name)

    mask   = labels != -1   # exclude HDBSCAN noise from the cluster analysis
    values = data["P"][mask, pi]
    labels = labels[mask]

    cluster_ids   = np.unique(labels)
    cluster_means = np.array([values[labels == cid].mean() for cid in cluster_ids])
    order         = np.argsort(cluster_means)
    cluster_ids   = cluster_ids[order]
    cluster_means = cluster_means[order]

    vmin, vmax = np.percentile(values, 1), np.percentile(values, 99)
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap("bwr")
    x_grid = np.linspace(vmin, vmax, 500)

    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 0.9 * len(cluster_ids) + 2))

    overlap = 0.7   # vertical spacing between ridges (in normalized density units)
    for i, (cid, mean_val) in enumerate(zip(cluster_ids, cluster_means)):
        vals_c  = values[labels == cid]
        density = gaussian_kde(vals_c)(x_grid)
        density = density / density.max()

        base_y = i * overlap
        color  = cmap(norm(mean_val))

        ax.fill_between(x_grid, base_y, base_y + density, color=color, alpha=0.85, zorder=-i)
        ax.plot(x_grid, base_y + density, color="black", linewidth=0.6, zorder=-i)
        ax.axhline(base_y, color="gray", linewidth=0.4, zorder=-i - 1)
        ax.text(vmin, base_y + 0.05, f"Cluster {cid}  (n={len(vals_c)})", fontsize=8, va="bottom")

    ax.set_yticks([])
    ax.set_xlim(vmin, vmax)
    ax.set_xlabel(inv_name)
    ax.set_title(
        f"{model_name} — {inv_name} distribution per cluster\n"
        "(ranked by mean, colour = position on the invariant axis; HDBSCAN noise excluded)",
        fontsize=12, fontweight="bold",
    )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label=inv_name, fraction=0.03, pad=0.02)

    return ax


plot_invariant_ridgeline(RIDGE_MODEL, RIDGE_INVARIANT)
plt.show()


In [ ]:
# ── 4-3  Aggregated view: intra-cluster vs. global variance ratio ─────────────
# For each (setup × invariant): mean intra-cluster variance (averaged over that
# setup's clusters) divided by the global variance of that invariant. Low
# (green) → clusters are compact / well separated along this invariant. High
# (red, ≈ 1) → clustering does not isolate this invariant at all. HDBSCAN noise
# points (-1) are excluded from both the intra-cluster and the global variance.

variance_ratio_rows = []

for model_name in CLUSTER_MODELS:
    data   = cluster_data[model_name]
    labels = data["labels"]
    P      = data["P"]

    mask      = labels != -1   # exclude HDBSCAN noise
    labels_nz = labels[mask]
    P_nz      = P[mask]

    for inv_name in TSNE_INVARIANTS:
        pi     = physical_variable_names.index(inv_name)
        values = P_nz[:, pi]

        global_var     = values.var()
        intra_vars     = [values[labels_nz == cid].var() for cid in np.unique(labels_nz)]
        mean_intra_var = np.mean(intra_vars)

        variance_ratio_rows.append({
            "model": model_name,
            "invariant": inv_name,
            "variance_ratio": mean_intra_var / global_var if global_var > 0 else np.nan,
        })

variance_ratio_df = pd.DataFrame(variance_ratio_rows)
variance_ratio_pivot = (variance_ratio_df
                        .pivot(index="invariant", columns="model", values="variance_ratio")
                        .loc[TSNE_INVARIANTS, CLUSTER_MODELS])

fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
im = ax.imshow(variance_ratio_pivot.values, cmap="RdYlGn_r", vmin=0, vmax=1, aspect="auto")

ax.set_xticks(np.arange(len(CLUSTER_MODELS)))
ax.set_xticklabels(CLUSTER_MODELS, rotation=45, ha="right")
ax.set_yticks(np.arange(len(TSNE_INVARIANTS)))
ax.set_yticklabels(TSNE_INVARIANTS)

for i in range(variance_ratio_pivot.shape[0]):
    for j in range(variance_ratio_pivot.shape[1]):
        ax.text(j, i, f"{variance_ratio_pivot.values[i, j]:.2f}",
                ha="center", va="center", color="black", fontsize=9)

plt.colorbar(im, ax=ax, label="Mean intra-cluster variance / global variance")
ax.set_title(
    "Intra-cluster vs. global variance ratio  (HDBSCAN, noise excluded)\n"
    "Low (green) = clusters strongly separate on this invariant  |  High (red, ≈1) = clustering doesn't isolate it",
    fontsize=12,
)
plt.show()


In [ ]:
# ── 4-4  Ridgeline per cluster, split by climate ──────────────────────────────
# Same idea as 4-2, but instead of a single distribution per cluster, each
# cluster shows one distribution per climate (coloured with climate_colors),
# so we can check whether an invariant's distribution within a cluster depends
# on climate. Uses the same RIDGE_MODEL / RIDGE_INVARIANT chosen in cell 4-2.
# HDBSCAN noise points (-1) are excluded.

def plot_invariant_ridgeline_by_climate(model_name, inv_name, ax=None):
    data   = cluster_data[model_name]
    labels = data["labels"]
    clim   = data["clim"]
    pi     = physical_variable_names.index(inv_name)

    mask   = labels != -1   # exclude HDBSCAN noise
    values = data["P"][mask, pi]
    labels = labels[mask]
    clim   = clim[mask]

    cluster_ids   = np.unique(labels)
    cluster_means = np.array([values[labels == cid].mean() for cid in cluster_ids])
    cluster_ids   = cluster_ids[np.argsort(cluster_means)]

    vmin, vmax = np.percentile(values, 1), np.percentile(values, 99)
    x_grid = np.linspace(vmin, vmax, 500)

    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 1.4 * len(cluster_ids) + 2))

    overlap = 1.0   # more vertical room than 4-2 since several climate curves share a ridge
    for i, cid in enumerate(cluster_ids):
        base_y       = i * overlap
        cluster_mask = labels == cid
        n_cluster    = int(cluster_mask.sum())

        for c in climate_order:
            vals_c = values[cluster_mask & (clim == c)]
            if len(vals_c) < 2:
                continue
            density = gaussian_kde(vals_c)(x_grid)
            density = density / density.max() * 0.9

            color = climate_colors.get(c, None)
            ax.plot(x_grid, base_y + density, color=color, linewidth=1.3, label=c)
            ax.fill_between(x_grid, base_y, base_y + density, color=color, alpha=0.15)

        ax.axhline(base_y, color="gray", linewidth=0.4, zorder=-i - 1)
        ax.text(vmin, base_y + 0.95, f"Cluster {cid}  (n={n_cluster})", fontsize=8, va="bottom")

    ax.set_yticks([])
    ax.set_xlim(vmin, vmax)
    ax.set_xlabel(inv_name)
    ax.set_title(
        f"{model_name} — {inv_name} distribution per cluster, split by climate\n"
        "(clusters ranked by overall mean; HDBSCAN noise excluded)",
        fontsize=12, fontweight="bold",
    )

    handles, labels_leg = ax.get_legend_handles_labels()
    by_label = dict(zip(labels_leg, handles))   # de-duplicate repeated climate labels across clusters
    ax.legend(by_label.values(), by_label.keys(), frameon=False, fontsize=8, loc="upper right")

    return ax


plot_invariant_ridgeline_by_climate(RIDGE_MODEL, RIDGE_INVARIANT)
plt.show()


## Part 5 - Analyzing cross climate neighbour

It should be noted that, up to this point, the study of invariants in latent representations has been carried out by using, for each invariant, the mean value of that invariant over a given sample and assigning this average value to the corresponding point in the latent space. This initial analysis is somewhat simplistic; therefore, in this section and the next, we choose to consider a vector that provides a more representative description of the distribution within each sample.

For each sample $x_i$, each invariant $I$, and each spatial patch, we compute:

$$
s_I(x_i)
=
\left[
\mu,\,
\sigma,\,
q_{10},\,
q_{50},\,
q_{90},\,
\|\nabla I\|_{\mathrm{mean}},\,
\|\nabla I\|_{\mathrm{max}}
\right].
$$

In [ ]:
# ── 5-0  Summary-vector extraction (V2 of extract_test_latent_and_physical) ───
# Instead of collapsing each (sample, physical invariant) pair to a single mean
# value, this returns a 7-value vector per (sample, invariant):
#   [mean, std, q10, q50, q90, ||grad I||_mean, ||grad I||_max]
# The gradient is computed on the patch's native 10 (lat) x 7 (lon) grid
# (confirmed from build_multivariate_samples_optimized.py and
# CMIP_mask_exp5_CERA.ipynb: flat index = lat_idx * n_lon + lon_idx, i.e. a
# plain `reshape(n_lat, n_lon)` in row-major order), using simple finite
# differences (arbitrary units — sufficient for this relative comparison).

n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
assert n_lat * n_lon == grid_points_per_patch

SUMMARY_STAT_NAMES = ["mean", "std", "q10", "q50", "q90", "grad_mean", "grad_max"]


def _compute_summary_stats(flat_block, n_vars, grid_points_per_patch, n_lat, n_lon):
    """flat_block: (N, n_vars * grid_points_per_patch), contiguous per-variable blocks.
    Returns (N, n_vars, len(SUMMARY_STAT_NAMES))."""
    N = flat_block.shape[0]
    block = np.asarray(flat_block, dtype=np.float64).reshape(N, n_vars, grid_points_per_patch)

    mean = block.mean(axis=2)
    std  = block.std(axis=2)
    q10  = np.percentile(block, 10, axis=2)
    q50  = np.percentile(block, 50, axis=2)
    q90  = np.percentile(block, 90, axis=2)

    grid = block.reshape(N, n_vars, n_lat, n_lon)
    dlat, dlon = np.gradient(grid, axis=(2, 3))
    grad_mag  = np.sqrt(dlat ** 2 + dlon ** 2)
    grad_mean = grad_mag.mean(axis=(2, 3))
    grad_max  = grad_mag.max(axis=(2, 3))

    return np.stack([mean, std, q10, q50, q90, grad_mean, grad_max], axis=-1).astype(np.float32)


def extract_test_latent_and_physical_v2(latent_key, n_per_climate, seed=42):
    """Like extract_test_latent_and_physical, but returns a 7-value summary vector
    per (sample, physical invariant) instead of just the mean.

    Z    : (n_total, latent_dim)                                     latent vectors
    S    : (n_total, n_physical_variables, len(SUMMARY_STAT_NAMES))  summary vectors
    clim : list of climate labels, one per sample
    """
    rng = np.random.default_rng(seed)
    Z_parts, S_parts, clim_parts = [], [], []

    for c in climate_order:
        Z_all = latent_test_sets[latent_key]["latent_test_by_climate"][c]

        aug = augmented_features_by_climate[c]
        if c == "historical":
            aug = aug[ae_split_indices["historical"]["test"]]

        P_flat = aug[:, _n_raw_cols:]

        reindex = reindex_maps[latent_key][c]
        if reindex is not None:
            P_flat = P_flat[reindex]

        n_avail = len(Z_all)
        n = min(n_per_climate, n_avail)
        idx = rng.choice(n_avail, size=n, replace=False)

        S_sub = _compute_summary_stats(P_flat[idx], n_physical_variables, grid_points_per_patch, n_lat, n_lon)

        Z_parts.append(np.asarray(Z_all[idx], dtype=np.float32))
        S_parts.append(S_sub)
        clim_parts.extend([c] * n)

    return np.vstack(Z_parts), np.concatenate(S_parts, axis=0), clim_parts


print(f"Grid geometry: n_lat={n_lat}, n_lon={n_lon}  ->  summary stats: {SUMMARY_STAT_NAMES}")


In [ ]:
# ── 5-1  Build cross-climate nearest-neighbour pairs for 6 datasets ───────────
# 6 pairing methods, all starting from the SAME fixed pool of historical anchor
# samples:
#   CERA      : nearest neighbour in the 48 aligned dims of the exp5_cera latent
#   Baseline  : nearest neighbour in the first 48 dims of the exp5_baseline2 latent
#   exp3      : nearest neighbour in the full 64D exp3_AEall latent
#   ClimaX    : nearest neighbour in the full 64D exp5_baseline_climax latent
#   RawData   : nearest neighbour in a 105D (15 raw vars x 7 stats) summary space
#   Random    : a uniformly random SSP sample (any climate) per anchor
#
# Sample identity is tracked via the (time, patch_id) key (see
# "Metadata alignment verification" above) rather than raw array position,
# since different latent files are not guaranteed to store samples in the same
# order — this guarantees "anchor #k" refers to the exact same physical sample
# across all 6 datasets.

from sklearn.neighbors import NearestNeighbors

N_PAIRS_TARGET = 10000
PAIR_SEED = random_seed + 500
_pair_rng = np.random.default_rng(PAIR_SEED)

ssp_climates = [c for c in climate_order if c != "historical"]

PAIR_LATENT_KEYS = {
    "CERA":     ("exp5_cera", 48),
    "Baseline": ("exp5_baseline2", 48),
    "SWDN":     ("exp5_cera_swdn", 48),
    "ClimaX":   ("exp5_baseline_climax", None),
}

# ── Canonical historical anchor set (identical across all 6 datasets) ────────
hist_test_idx  = ae_split_indices["historical"]["test"]
hist_meta_test = metadata_by_climate["historical"].iloc[hist_test_idx].reset_index(drop=True)
hist_keys_all  = _sample_keys(hist_meta_test)
n_avail_hist   = len(hist_keys_all)
N_PAIRS        = min(N_PAIRS_TARGET, n_avail_hist)

anchor_pos      = _pair_rng.choice(n_avail_hist, size=N_PAIRS, replace=False)
anchor_keys     = hist_keys_all[anchor_pos]
anchor_hist_idx = hist_test_idx[anchor_pos]   # absolute row index into metadata_by_climate["historical"]

print(f"Anchor pool: {n_avail_hist} historical test points available -> using {N_PAIRS} anchors "
      f"(target was {N_PAIRS_TARGET}).")


# ── RawData baseline: 105D (15 raw vars x 7 stats) nearest-neighbour ─────────
def _raw_input_summary(climate):
    aug = augmented_features_by_climate[climate]
    if climate == "historical":
        aug = aug[ae_split_indices["historical"]["test"]]
    X_flat = aug[:, :_n_raw_cols]
    return _compute_summary_stats(X_flat, n_input_variables, grid_points_per_patch, n_lat, n_lon)


raw_hist_summary_all    = _raw_input_summary("historical").reshape(n_avail_hist, -1)   # (n_avail_hist, 105)
raw_hist_summary_anchor = raw_hist_summary_all[anchor_pos]                              # (N_PAIRS, 105)

raw_ssp_summary_parts, raw_ssp_climate_parts, raw_ssp_index_parts = [], [], []
for c in ssp_climates:
    S = _raw_input_summary(c).reshape(len(metadata_by_climate[c]), -1)
    raw_ssp_summary_parts.append(S)
    raw_ssp_climate_parts.append(np.full(len(S), c))
    raw_ssp_index_parts.append(np.arange(len(S)))

raw_ssp_summary_pool = np.vstack(raw_ssp_summary_parts)
raw_ssp_pool_climate = np.concatenate(raw_ssp_climate_parts)
raw_ssp_pool_index   = np.concatenate(raw_ssp_index_parts)

nn_raw = NearestNeighbors(n_neighbors=1).fit(raw_ssp_summary_pool)
_, raw_nn_pos = nn_raw.kneighbors(raw_hist_summary_anchor)
raw_nn_pos = raw_nn_pos[:, 0]


# ── Random baseline ───────────────────────────────────────────────────────────
random_target_climate = _pair_rng.choice(ssp_climates, size=N_PAIRS)
random_ssp_index = np.array([
    _pair_rng.integers(0, len(metadata_by_climate[c])) for c in random_target_climate
])


# ── The 4 latent-based nearest-neighbour methods ─────────────────────────────
def build_latent_pairs(latent_key, slice_dim):
    hist_lmeta = pd.DataFrame(latent_test_sets[latent_key]["latent_test_metadata_by_climate"]["historical"])
    hist_lkeys = _sample_keys(hist_lmeta)
    key_to_row = {k: i for i, k in enumerate(hist_lkeys)}

    missing = [k for k in anchor_keys if k not in key_to_row]
    if missing:
        raise ValueError(f"[{latent_key}] {len(missing)} anchor keys missing from its historical latent block.")

    hist_rows  = np.array([key_to_row[k] for k in anchor_keys])
    Z_hist_all = np.asarray(latent_test_sets[latent_key]["latent_test_by_climate"]["historical"])
    Z_hist     = Z_hist_all[hist_rows]
    if slice_dim is not None:
        Z_hist = Z_hist[:, :slice_dim]

    Z_ssp_parts, ssp_pool_climate_parts, ssp_pool_index_parts = [], [], []
    for c in ssp_climates:
        Zc = np.asarray(latent_test_sets[latent_key]["latent_test_by_climate"][c])
        if slice_dim is not None:
            Zc = Zc[:, :slice_dim]

        lmeta_c        = pd.DataFrame(latent_test_sets[latent_key]["latent_test_metadata_by_climate"][c])
        keys_c         = _sample_keys(lmeta_c)
        raw_key_to_row = {k: i for i, k in enumerate(_sample_keys(metadata_by_climate[c]))}
        idx_in_raw     = np.array([raw_key_to_row[k] for k in keys_c])

        Z_ssp_parts.append(Zc)
        ssp_pool_climate_parts.append(np.full(len(Zc), c))
        ssp_pool_index_parts.append(idx_in_raw)

    Z_ssp_pool       = np.vstack(Z_ssp_parts)
    ssp_pool_climate = np.concatenate(ssp_pool_climate_parts)
    ssp_pool_index   = np.concatenate(ssp_pool_index_parts)

    nn = NearestNeighbors(n_neighbors=1).fit(Z_ssp_pool)
    _, nn_pos = nn.kneighbors(Z_hist)
    nn_pos = nn_pos[:, 0]

    return ssp_pool_climate[nn_pos], ssp_pool_index[nn_pos]


pair_results = {
    "Random":  (random_target_climate, random_ssp_index),
    "RawData": (raw_ssp_pool_climate[raw_nn_pos], raw_ssp_pool_index[raw_nn_pos]),
}
for setup_name, (latent_key, slice_dim) in PAIR_LATENT_KEYS.items():
    print(f"Building nearest-neighbour pairs for {setup_name}…")
    pair_results[setup_name] = build_latent_pairs(latent_key, slice_dim)

print("\nPairs built:")
for setup_name, (tclim, sidx) in pair_results.items():
    climate_counts = dict(zip(*np.unique(tclim, return_counts=True)))
    print(f"  {setup_name:10s}: {len(tclim)} pairs — target climate counts = {climate_counts}")


In [ ]:
# ── 5-2  Delta computation: build the final long-format dataframe ────────────
# For each dataset (pairing method) x pair x invariant x summary statistic:
#     delta = |s(x_hist) - s(x_ssp)| / std(s over the N_PAIRS historical anchors)
# The normalising std only depends on (invariant, summary_stat) — it is the
# SAME for all 6 datasets since they all share the same historical anchors.

def _physical_invariant_summary(climate):
    aug = augmented_features_by_climate[climate]
    if climate == "historical":
        aug = aug[ae_split_indices["historical"]["test"]]
    P_flat = aug[:, _n_raw_cols:]
    return _compute_summary_stats(P_flat, n_physical_variables, grid_points_per_patch, n_lat, n_lon)


hist_invariant_summary_all    = _physical_invariant_summary("historical")          # (n_avail_hist, 25, 7)
hist_invariant_summary_anchor = hist_invariant_summary_all[anchor_pos]             # (N_PAIRS, 25, 7)

ssp_invariant_summary_by_clim = {c: _physical_invariant_summary(c) for c in ssp_climates}

# Normalising std: same for all 6 datasets (same historical anchors everywhere)
std_norm      = hist_invariant_summary_anchor.std(axis=0)              # (25, 7)
std_norm_safe = np.where(std_norm > 0, std_norm, 1.0)

n_inv, n_stat = len(physical_variable_names), len(SUMMARY_STAT_NAMES)

delta_dfs = []
for setup_name, (target_climate, ssp_index) in pair_results.items():
    ssp_summary = np.stack([
        ssp_invariant_summary_by_clim[c][i] for c, i in zip(target_climate, ssp_index)
    ])   # (N_PAIRS, 25, 7)

    delta = np.abs(hist_invariant_summary_anchor - ssp_summary) / std_norm_safe   # (N_PAIRS, 25, 7)

    setup_df = pd.DataFrame({
        "setup_name":     np.repeat(setup_name, N_PAIRS * n_inv * n_stat),
        "target_climate": np.repeat(target_climate, n_inv * n_stat),
        "hist_index":     np.repeat(anchor_hist_idx, n_inv * n_stat),
        "ssp_index":      np.repeat(ssp_index, n_inv * n_stat),
        "invariant":      np.tile(np.repeat(physical_variable_names, n_stat), N_PAIRS),
        "summary_stat":   np.tile(SUMMARY_STAT_NAMES, N_PAIRS * n_inv),
        "delta_value":    delta.reshape(-1),
    })
    delta_dfs.append(setup_df)
    print(f"  {setup_name} done ({len(setup_df)} rows).")

pair_delta_df = pd.concat(delta_dfs, ignore_index=True)
for col in ["setup_name", "target_climate", "invariant", "summary_stat"]:
    pair_delta_df[col] = pair_delta_df[col].astype("category")

print(f"\nFinal dataframe: {pair_delta_df.shape}")
pair_delta_df.head()


In [ ]:
# ── 5-3  SSP composition of each dataset's pairs ──────────────────────────────
# For each of the 6 pairing methods, what proportion of the paired SSP points
# come from each SSP climate? A method that isn't climate-biased should show
# roughly uniform proportions across the 4 SSPs, similar to Random.

ssp_proportion_rows = []
for setup_name, (target_climate, _) in pair_results.items():
    counts = pd.Series(target_climate).value_counts(normalize=True).reindex(ssp_climates, fill_value=0.0)
    for c, p in counts.items():
        ssp_proportion_rows.append({"setup_name": setup_name, "target_climate": c, "proportion": p})

ssp_proportion_df = pd.DataFrame(ssp_proportion_rows)
ssp_proportion_pivot = (ssp_proportion_df
                         .pivot(index="setup_name", columns="target_climate", values="proportion")
                         .reindex(index=list(pair_results.keys()), columns=ssp_climates))

display(
    ssp_proportion_pivot.style
    .format("{:.1%}")
    .background_gradient(axis=1, cmap="YlGnBu")
    .set_caption("Proportion of each SSP climate among the paired SSP points, per dataset")
)

# ── Bar chart ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
setups = list(pair_results.keys())
x     = np.arange(len(setups))
width = 0.8 / len(ssp_climates)

for i, c in enumerate(ssp_climates):
    color = climate_colors.get(c, f"C{i}")
    ax.bar(x + (i - (len(ssp_climates) - 1) / 2) * width, ssp_proportion_pivot[c], width,
           color=color, alpha=0.9, label=c)

ax.axhline(1 / len(ssp_climates), color="gray", linestyle="--", linewidth=1, label="Uniform (1/4)")
ax.set_xticks(x)
ax.set_xticklabels(setups)
ax.set_ylabel("Proportion of pairs")
ax.set_ylim(0, 1)
ax.set_title(
    "SSP composition of the paired points, per dataset\n"
    "Flat across SSPs (~25% each) → no systematic bias toward a specific SSP"
)
ax.legend(frameon=False, ncol=len(ssp_climates) + 1, fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
# ── 5-4  Macro box plot: delta distribution per setup (distribution stats only) ─
# One box per setup, built from delta_value restricted to the 5 "distribution"
# summary stats (mean, std, q10, q50, q90) — excluding grad_mean/grad_max, which
# capture spatial texture rather than distributional similarity. Outliers are
# hidden (showfliers=False) so the boxes stay readable — with heavy-tailed
# quantities a handful of outliers can otherwise dominate the y-scale and hide
# the median/IQR differences between setups.

DISTRIBUTION_STATS = ["mean", "std", "q10", "q50", "q90"]

PAIR_SETUP_COLORS = {
    "Random":   "#B0B0B0",
    "RawData":  "#16A085",
    "CERA":     "#5B8DB8",
    "Baseline": "#E07B39",
    "ClimaX":   "#9B59B6",
}

dist_delta_df = pair_delta_df[pair_delta_df["summary_stat"].isin(DISTRIBUTION_STATS)]

setup_order = list(pair_results.keys())
box_data = [dist_delta_df.loc[dist_delta_df["setup_name"] == s, "delta_value"].values for s in setup_order]

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
bp = ax.boxplot(
    box_data,
    tick_labels=setup_order,
    showfliers=False,
    patch_artist=True,
    medianprops=dict(color="black", linewidth=1.5),
)

for patch, s in zip(bp["boxes"], setup_order):
    patch.set_facecolor(PAIR_SETUP_COLORS.get(s, "#5B8DB8"))
    patch.set_alpha(0.85)

ax.set_ylabel("delta_value  (|s(hist) − s(ssp)| / std)")
ax.set_title(
    "Macro comparison of paired-point similarity, per setup\n"
    "Distribution stats only (mean / std / q10 / q50 / q90) over all 25 invariants — outliers hidden\n"
    "Lower = SSP neighbour more physically similar to its historical anchor"
)
ax.grid(axis="y", alpha=0.3)
plt.show()

# ── Summary table: median / IQR per setup ─────────────────────────────────────
summary_rows = []
for s in setup_order:
    vals = dist_delta_df.loc[dist_delta_df["setup_name"] == s, "delta_value"]
    summary_rows.append({
        "setup_name": s,
        "median": vals.median(),
        "q25":    vals.quantile(0.25),
        "q75":    vals.quantile(0.75),
        "IQR":    vals.quantile(0.75) - vals.quantile(0.25),
    })

display(
    pd.DataFrame(summary_rows).set_index("setup_name")
    .style.format("{:.3f}")
    .background_gradient(subset=["median"], cmap="RdYlGn_r")
    .set_caption("Median / IQR of delta_value per setup (distribution stats only)")
)


In [ ]:
# ── 5-5  Heatmap: median delta per (setup x statistic), for a chosen invariant ─
# Choose an invariant below. For that invariant, compute the median delta_value
# across all pairs, for each (setup, summary_stat) combination, and display as
# a heatmap: setup in x, the 7 summary statistics in y.

HEATMAP_INVARIANT = "RH850"   # choose one of physical_variable_names

inv_df = pair_delta_df[pair_delta_df["invariant"] == HEATMAP_INVARIANT]

setup_order = list(pair_results.keys())
heatmap_pivot = (inv_df
                  .groupby(["summary_stat", "setup_name"])["delta_value"]
                  .median()
                  .unstack("setup_name")
                  .reindex(index=SUMMARY_STAT_NAMES, columns=setup_order))

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
im = ax.imshow(heatmap_pivot.values, cmap="RdYlGn_r", aspect="auto")

ax.set_xticks(np.arange(len(setup_order)))
ax.set_xticklabels(setup_order, rotation=45, ha="right")
ax.set_yticks(np.arange(len(SUMMARY_STAT_NAMES)))
ax.set_yticklabels(SUMMARY_STAT_NAMES)

for i in range(heatmap_pivot.shape[0]):
    for j in range(heatmap_pivot.shape[1]):
        ax.text(j, i, f"{heatmap_pivot.values[i, j]:.2f}",
                ha="center", va="center", color="black", fontsize=9)

plt.colorbar(im, ax=ax, label="Median delta_value")
ax.set_title(
    f"Median delta per setup × summary statistic — invariant: {HEATMAP_INVARIANT}\n"
    "Lower (green) = paired SSP point more similar to its historical anchor for this statistic",
    fontsize=12,
)
plt.show()


In [ ]:
# ── 5-6  Heatmap: mean delta per (setup x invariant), averaged over all pairs and statistics ─
# Unlike 5-5 (single invariant, per-statistic breakdown), this aggregates over
# ALL 7 summary statistics (including the gradients) and all pairs, for every
# invariant — a broad overview: setup in x, invariant in y. Restricted to the
# physical invariants (RH, lapse rate, shear, wind) — the "_anom" anomaly
# variables are excluded here.

setup_order = list(pair_results.keys())
physical_only_invariants = [v for v in physical_variable_names if not v.endswith("_anom")]

heatmap_inv_pivot = (pair_delta_df
                      .groupby(["invariant", "setup_name"])["delta_value"]
                      .mean()
                      .unstack("setup_name")
                      .reindex(index=physical_only_invariants, columns=setup_order))

fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
im = ax.imshow(heatmap_inv_pivot.values, cmap="RdYlGn_r", aspect="auto")

ax.set_xticks(np.arange(len(setup_order)))
ax.set_xticklabels(setup_order, rotation=45, ha="right")
ax.set_yticks(np.arange(len(physical_only_invariants)))
ax.set_yticklabels(physical_only_invariants, fontsize=8)

for i in range(heatmap_inv_pivot.shape[0]):
    for j in range(heatmap_inv_pivot.shape[1]):
        ax.text(j, i, f"{heatmap_inv_pivot.values[i, j]:.2f}",
                ha="center", va="center", color="black", fontsize=7)

plt.colorbar(im, ax=ax, label="Mean delta_value")
ax.set_title(
    "Mean delta per setup × invariant, averaged over all pairs and all 7 statistics\n"
    "Physical invariants only (anomalies excluded) — lower (green) = paired SSP point "
    "more similar to its historical anchor",
    fontsize=12,
)
plt.show()


In [ ]:
# ── 5-7  Correlation between latent distance and invariant delta ─────────────
# For the 4 latent-based setups (CERA, Baseline, exp3, ClimaX — Random and
# RawData are excluded: they have no dedicated latent space to measure a
# distance in), compute for every pair the Euclidean distance between the two
# points in that setup's own latent space (the same space used to find the
# pair), then correlate it with the invariant delta (averaged over the 5
# distribution stats — mean/std/q10/q50/q90, gradients excluded). Physical
# invariants only (anomalies excluded).

LATENT_DIST_SETUPS = ["CERA", "Baseline", "ClimaX"]
DISTRIBUTION_STATS = ["mean", "std", "q10", "q50", "q90"]
dist_stat_indices = [SUMMARY_STAT_NAMES.index(s) for s in DISTRIBUTION_STATS]
physical_only_invariants = [v for v in physical_variable_names if not v.endswith("_anom")]


def latent_pair_distances(latent_key, slice_dim, target_climate, ssp_index):
    """Euclidean distance, in this latent (sliced to slice_dim if not None),
    between each historical anchor and its paired SSP point."""
    hist_lmeta = pd.DataFrame(latent_test_sets[latent_key]["latent_test_metadata_by_climate"]["historical"])
    hist_lkeys = _sample_keys(hist_lmeta)
    key_to_row = {k: i for i, k in enumerate(hist_lkeys)}
    hist_rows  = np.array([key_to_row[k] for k in anchor_keys])
    Z_hist_all = np.asarray(latent_test_sets[latent_key]["latent_test_by_climate"]["historical"])
    Z_hist     = Z_hist_all[hist_rows]

    Z_ssp = np.empty_like(Z_hist)
    for c in np.unique(target_climate):
        mask = target_climate == c
        raw_key_to_row = {k: i for i, k in enumerate(_sample_keys(metadata_by_climate[c]))}
        lmeta_c = pd.DataFrame(latent_test_sets[latent_key]["latent_test_metadata_by_climate"][c])
        keys_c  = _sample_keys(lmeta_c)
        raw_row_to_latent_row = {raw_key_to_row[k]: i for i, k in enumerate(keys_c)}
        Zc_all = np.asarray(latent_test_sets[latent_key]["latent_test_by_climate"][c])
        latent_rows = np.array([raw_row_to_latent_row[i] for i in ssp_index[mask]])
        Z_ssp[mask] = Zc_all[latent_rows]

    if slice_dim is not None:
        Z_hist = Z_hist[:, :slice_dim]
        Z_ssp  = Z_ssp[:, :slice_dim]

    return np.linalg.norm(Z_hist - Z_ssp, axis=1)


corr_matrix = np.empty((len(physical_only_invariants), len(LATENT_DIST_SETUPS)))

for j, setup_name in enumerate(LATENT_DIST_SETUPS):
    latent_key, slice_dim = PAIR_LATENT_KEYS[setup_name]
    target_climate, ssp_index = pair_results[setup_name]

    dist_vec = latent_pair_distances(latent_key, slice_dim, target_climate, ssp_index)

    ssp_summary = np.stack([
        ssp_invariant_summary_by_clim[c][i] for c, i in zip(target_climate, ssp_index)
    ])   # (N_PAIRS, 25, 7)
    delta = np.abs(hist_invariant_summary_anchor - ssp_summary) / std_norm_safe   # (N_PAIRS, 25, 7)
    delta_avg_stats = delta[:, :, dist_stat_indices].mean(axis=2)                 # (N_PAIRS, 25)

    for i, inv_name in enumerate(physical_only_invariants):
        pi = physical_variable_names.index(inv_name)
        corr_matrix[i, j] = np.corrcoef(dist_vec, delta_avg_stats[:, pi])[0, 1]

    print(f"  {setup_name}: latent distance mean={dist_vec.mean():.3f}, std={dist_vec.std():.3f}")

fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
im = ax.imshow(corr_matrix, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")

ax.set_xticks(np.arange(len(LATENT_DIST_SETUPS)))
ax.set_xticklabels(LATENT_DIST_SETUPS, rotation=45, ha="right")
ax.set_yticks(np.arange(len(physical_only_invariants)))
ax.set_yticklabels(physical_only_invariants, fontsize=8)

for i in range(corr_matrix.shape[0]):
    for j in range(corr_matrix.shape[1]):
        ax.text(j, i, f"{corr_matrix[i, j]:.2f}", ha="center", va="center", color="black", fontsize=8)

plt.colorbar(im, ax=ax, label="Pearson correlation (latent distance, invariant delta)")
ax.set_title(
    "Correlation between latent-space pair distance and invariant delta\n"
    "(delta averaged over mean/std/q10/q50/q90 — gradients excluded; physical invariants only)\n"
    "Higher (green) = latent distance is a more meaningful proxy for physical dissimilarity",
    fontsize=11,
)
plt.show()


In [ ]:
# ── 5-8  Clustering the pairs themselves (delta-profile clustering) ──────────
# For each of the 4 latent-based setups, build a matrix where each row is a
# pair and each column is one (physical invariant x summary stat) delta value
# — restricted to the 10 physical invariants (anomalies excluded) and 4
# statistics (mean, std, q10, q90 — median and gradients excluded) — then run
# HDBSCAN on this 40D "delta profile" space to see whether there are distinct
# modes of pairing behaviour (e.g. some pairs uniformly similar across all
# invariants, others similar on some but not others, etc.).

import hdbscan

PAIR_CLUSTER_SETUPS = ["CERA", "Baseline", "ClimaX"]
PAIR_CLUSTER_STATS  = ["mean", "std", "q10", "q90"]
pair_cluster_stat_indices = [SUMMARY_STAT_NAMES.index(s) for s in PAIR_CLUSTER_STATS]
physical_only_invariants  = [v for v in physical_variable_names if not v.endswith("_anom")]
physical_only_indices     = [physical_variable_names.index(v) for v in physical_only_invariants]

# Start modest given past experience tuning HDBSCAN on this project (Part 4):
# the library defaults (min_cluster_size=5) tend to surface tiny spurious
# clusters, while overly large values can collapse everything to noise.
# Tune these once you've seen the actual cluster sizes/noise fraction below.
PAIR_CLUSTER_MIN_CLUSTER_SIZE = 30
PAIR_CLUSTER_MIN_SAMPLES      = 10

pair_feature_columns = [f"{inv}__{stat}" for inv in physical_only_invariants for stat in PAIR_CLUSTER_STATS]

pair_cluster_results = {}   # {setup_name: {"labels", "features", "n_clusters", "n_noise"}}

print(f"Clustering pairs (delta-profile space) with HDBSCAN "
      f"(min_cluster_size={PAIR_CLUSTER_MIN_CLUSTER_SIZE}, min_samples={PAIR_CLUSTER_MIN_SAMPLES})…")
for setup_name in PAIR_CLUSTER_SETUPS:
    target_climate, ssp_index = pair_results[setup_name]
    ssp_summary = np.stack([
        ssp_invariant_summary_by_clim[c][i] for c, i in zip(target_climate, ssp_index)
    ])   # (N_PAIRS, 25, 7)
    delta = np.abs(hist_invariant_summary_anchor - ssp_summary) / std_norm_safe   # (N_PAIRS, 25, 7)

    # Restrict to physical-only invariants and the 4 chosen stats -> (N_PAIRS, 10, 4) -> (N_PAIRS, 40)
    features = delta[:, physical_only_indices, :][:, :, pair_cluster_stat_indices].reshape(len(delta), -1)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=PAIR_CLUSTER_MIN_CLUSTER_SIZE,
        min_samples=PAIR_CLUSTER_MIN_SAMPLES,
    )
    labels = clusterer.fit_predict(features)

    n_noise    = int((labels == -1).sum())
    n_clusters = len(set(labels.tolist())) - (1 if -1 in labels else 0)

    pair_cluster_results[setup_name] = {
        "labels": labels, "features": features, "n_clusters": n_clusters, "n_noise": n_noise,
    }
    sizes = sorted(np.bincount(labels[labels >= 0]), reverse=True) if n_clusters > 0 else []
    print(f"  {setup_name:10s}: clusters={n_clusters:3d}, noise={n_noise:6d} "
          f"({100 * n_noise / len(labels):.1f}%), cluster sizes={sizes}")


# ── Display 1: cluster sizes per setup ────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
axes = axes.flatten()

for ax, setup_name in zip(axes, PAIR_CLUSTER_SETUPS):
    res    = pair_cluster_results[setup_name]
    labels = res["labels"]
    sizes  = sorted(np.bincount(labels[labels >= 0]), reverse=True) if res["n_clusters"] > 0 else []

    bar_labels = [str(i) for i in range(len(sizes))] + (["noise"] if res["n_noise"] > 0 else [])
    bar_values = sizes + ([res["n_noise"]] if res["n_noise"] > 0 else [])
    bar_colors = ["#5B8DB8"] * len(sizes) + (["#B0B0B0"] if res["n_noise"] > 0 else [])

    ax.bar(bar_labels, bar_values, color=bar_colors, alpha=0.9)
    ax.set_title(f"{setup_name}  ({res['n_clusters']} clusters, {res['n_noise']} noise pts)",
                 fontsize=12, fontweight="bold")
    ax.set_ylabel("Number of pairs")
    ax.set_xlabel("Cluster (sorted by size, largest first)")
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Pair clustering (delta-profile space) — cluster sizes per setup", fontsize=14)
plt.show()


# ── Display 2: mean delta profile per cluster, split into 4 heatmaps ─────────
# Choose one setup below. The 40 columns (10 physical invariants x 4 stats)
# are split into 4 heatmaps (one per statistic, stacked vertically), each with
# 10 columns (the invariants) — x = invariant, y = cluster, colour = mean
# delta_value of that cluster for that (invariant, statistic). If more than 10
# clusters were found, nothing is plotted (too many rows to read) and a
# message is printed instead.

PAIR_CLUSTER_HEATMAP_SETUP = "CERA"   # choose one of PAIR_CLUSTER_SETUPS

res    = pair_cluster_results[PAIR_CLUSTER_HEATMAP_SETUP]
labels, features = res["labels"], res["features"]
cluster_ids = sorted(set(labels.tolist()) - {-1})

if not cluster_ids:
    print(f"{PAIR_CLUSTER_HEATMAP_SETUP}: no clusters found (all noise) — nothing to display.")
elif len(cluster_ids) > 10:
    print(f"{PAIR_CLUSTER_HEATMAP_SETUP}: {len(cluster_ids)} clusters found — more than 10, skipping display.")
else:
    profile = np.stack([features[labels == cid].mean(axis=0) for cid in cluster_ids])   # (n_clusters, 40)
    profile_3d = profile.reshape(len(cluster_ids), len(physical_only_invariants), len(PAIR_CLUSTER_STATS))

    vmin, vmax = profile.min(), profile.max()

    fig, axes = plt.subplots(len(PAIR_CLUSTER_STATS), 1,
                              figsize=(10, 3.2 * len(PAIR_CLUSTER_STATS)), constrained_layout=True)
    axes = np.atleast_1d(axes)

    for si, (ax, stat_name) in enumerate(zip(axes, PAIR_CLUSTER_STATS)):
        mat = profile_3d[:, :, si]   # (n_clusters, 10 invariants)
        im = ax.imshow(mat, cmap="RdYlGn_r", vmin=vmin, vmax=vmax, aspect="auto")

        ax.set_xticks(np.arange(len(physical_only_invariants)))
        ax.set_xticklabels(physical_only_invariants, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(np.arange(len(cluster_ids)))
        ax.set_yticklabels([f"Cluster {cid}  (n={int((labels == cid).sum())})" for cid in cluster_ids], fontsize=8)
        ax.set_title(f"statistic = {stat_name}", fontsize=10, fontweight="bold")
        plt.colorbar(im, ax=ax, label="Mean delta_value")

    fig.suptitle(
        f"{PAIR_CLUSTER_HEATMAP_SETUP} — mean delta profile per cluster, split by statistic\n"
        "x = physical invariant, y = cluster, colour = mean delta_value in cluster",
        fontsize=12,
    )
    plt.show()


## Part 6 - Search for emerging invariants

The objective of this final experiment is to identify which physical quantities remain approximately conserved between climate states that are considered analogous by the latent representation.

Given a pair of neighboring latent representations

$$
(z_i^{hist}, z_j^{ssp}),
$$

we assume that the model implicitly considers the corresponding physical states

$$
(x_i^{hist}, x_j^{ssp})
$$

to be similar. We therefore seek a function

$$
g(x)
$$

such that

$$
g(x_i^{hist}) \approx g(x_j^{ssp})
$$

for latent-neighbor pairs. If such a function can be identified, it may reveal a physical invariant implicitly learned by the model.

### Global Invariant Search

We first perform the analysis using all latent-neighbor pairs identified in the previous section, assuming that a single invariant may explain a large fraction of the latent organization.

Two complementary approaches are considered:

#### 1. Sparse Linear Regression (LASSO)

We construct a library of candidate physical quantities derived from the original variables, including raw variables, anomalies, ratios, gradients, and thermodynamic quantities.

We then search for a function of the form

$$
g(x) = \sum_k \alpha_k \phi_k(x),
$$

where $\phi_k(x)$ are candidate physical features.

LASSO (Least Absolute Shrinkage and Selection Operator) introduces an $L_1$ regularization term that promotes sparsity in the coefficients:

$$
\mathcal{L}
=
\sum_{(i,j)}
\left(g(x_i)-g(x_j)\right)^2
+
\lambda \sum_k |\alpha_k|.
$$

As a result, only a small number of physical features are selected, yielding an interpretable candidate invariant.

#### 2. Symbolic Regression

While LASSO is restricted to linear combinations of predefined features, symbolic regression searches directly for an analytical expression that remains approximately constant between latent-neighbor pairs.

The method explores a space of mathematical expressions built from the available physical variables and operators in order to discover a compact expression

$$
g(x)
$$

such that

$$
g(x_i^{hist}) \approx g(x_j^{ssp}).
$$

This approach may recover known invariants or reveal previously unidentified physical relationships.

### Local Invariant Search

The global analysis implicitly assumes that a single invariant explains the entire latent space. However, the latent representation may encode multiple physical mechanisms simultaneously.

The analyses performed in the previous sections suggest that the latent space does not naturally decompose into well-separated geometric clusters. Instead, different families of latent-neighbor pairs can be identified based on the physical invariants they preserve.

We therefore repeat the previous procedure separately on the clusters of latent-neighbor pairs obtained from the invariant-based analysis of Part 5.

For each cluster of analogues, we perform:

1. LASSO regression,
2. Symbolic regression.

This allows the identification of local invariants that may characterize specific physical regimes. Rather than searching for a single universal climate invariant, this approach aims at discovering a collection of regime-dependent invariants that jointly explain the structure of the latent space.

### Global Invariant Search

In [ ]:
# ── 6-0  Global Invariant Search — LASSO (SFA-style) ──────────────────────────
# For each of the 4 latent-based setups, look for a sparse linear combination g
# of a bank of known quantities (25 invariants + 15 raw variables, mean only =
# 40 candidates) that stays approximately constant across that setup's
# historical->SSP pairs, while still varying meaningfully across the whole
# dataset (Slow Feature Analysis-style objective):
#     minimise   E[(g(hist) - g(ssp))^2]
#     subject to Var(g(x)) > eps   (over all historical + SSP points used)
#
# This isn't a plain sklearn.Lasso.fit(X, y) problem — there is no external
# target y. We first solve the generalized eigenvalue problem that gives the
# exact (dense) solution to this constrained objective, then sparsify it by
# fitting a Lasso regression that reconstructs that dense direction's
# projection from the same bank of features. This gives an actual sklearn
# Lasso model whose coefficients can be inspected.

from sklearn.linear_model import LassoCV
from scipy.linalg import eigh

LASSO_SETUPS = ["CERA", "Baseline", "ClimaX"]
RIDGE_EPS = 1e-6     # numerical stabiliser added to the pooled covariance matrix
# Sparsification strength for the second-stage Lasso fit is now selected
# automatically per setup via cross-validation (LassoCV) instead of a fixed
# alpha — a hardcoded alpha=0.01 was collapsing every coefficient to zero on
# real data (alpha exceeded alpha_max for this problem's scale).
LASSO_CV_FOLDS = 5

bank_columns = (
    [f"{inv}__mean" for inv in physical_variable_names] +
    [f"{var}__mean" for var in selected_variables]
)
n_bank = len(bank_columns)
print(f"Bank size: {n_bank} candidate variables "
      f"({len(physical_variable_names)} invariants + {len(selected_variables)} raw vars, mean only)")


def _raw_input_summary_full(climate):
    """7-stat summary vector for all 15 raw input variables (all stats)."""
    aug = augmented_features_by_climate[climate]
    if climate == "historical":
        aug = aug[ae_split_indices["historical"]["test"]]
    X_flat = aug[:, :_n_raw_cols]
    return _compute_summary_stats(X_flat, n_input_variables, grid_points_per_patch, n_lat, n_lon)


raw_hist_summary_full        = _raw_input_summary_full("historical")        # (n_avail_hist, 15, 7)
raw_hist_summary_full_anchor = raw_hist_summary_full[anchor_pos]            # (N_PAIRS, 15, 7)
raw_ssp_summary_full_by_clim = {c: _raw_input_summary_full(c) for c in ssp_climates}


mean_idx = SUMMARY_STAT_NAMES.index("mean")


def bank_vectors(invariant_summary, raw_summary):
    """Select the mean statistic only from (invariants, 7) and (raw vars, 7)
    into one (N, n_bank) matrix — n_bank = n_invariants + n_raw_vars."""
    return np.concatenate([
        invariant_summary[:, :, mean_idx],   # (N, 25) — mean only
        raw_summary[:, :, mean_idx],          # (N, 15) — mean only
    ], axis=1)


bank_hist_anchor = bank_vectors(hist_invariant_summary_anchor, raw_hist_summary_full_anchor)  # (N_PAIRS, n_bank)

lasso_models = {}   # {setup_name: {"model", "coefs", "eigval"}}

print("Fitting SFA direction + sparsifying Lasso per setup…")
for setup_name in LASSO_SETUPS:
    target_climate, ssp_index = pair_results[setup_name]

    ssp_inv_summary = np.stack([
        ssp_invariant_summary_by_clim[c][i] for c, i in zip(target_climate, ssp_index)
    ])
    ssp_raw_summary = np.stack([
        raw_ssp_summary_full_by_clim[c][i] for c, i in zip(target_climate, ssp_index)
    ])
    bank_ssp = bank_vectors(ssp_inv_summary, ssp_raw_summary)   # (N_PAIRS, n_bank)

    diff   = bank_hist_anchor - bank_ssp                        # (N_PAIRS, n_bank)
    pooled = np.vstack([bank_hist_anchor, bank_ssp])            # (2*N_PAIRS, n_bank)

    pooled_mean = pooled.mean(axis=0)
    pooled_std  = pooled.std(axis=0)
    pooled_std_safe = np.where(pooled_std > 0, pooled_std, 1.0)

    diff_scaled   = diff / pooled_std_safe
    pooled_scaled = (pooled - pooled_mean) / pooled_std_safe

    C_diff   = (diff_scaled.T @ diff_scaled) / len(diff_scaled)
    C_pooled = (pooled_scaled.T @ pooled_scaled) / len(pooled_scaled)
    C_pooled += RIDGE_EPS * np.eye(n_bank)

    # Generalised eigenvalue problem: minimise w^T C_diff w s.t. w^T C_pooled w = 1
    # -> take the eigenvector with the SMALLEST generalised eigenvalue.
    eigvals, eigvecs = eigh(C_diff, C_pooled)
    w_dense = eigvecs[:, 0]
    eigval  = eigvals[0]

    # Sparsify: LassoCV reconstructing the dense SFA projection from the same
    # bank, automatically selecting alpha per setup via cross-validation.
    g_dense = pooled_scaled @ w_dense
    lasso = LassoCV(cv=LASSO_CV_FOLDS, fit_intercept=False, max_iter=50000, n_jobs=-1)
    lasso.fit(pooled_scaled, g_dense)

    lasso_models[setup_name] = {
        "model": lasso, "coefs": lasso.coef_, "eigval": eigval, "alpha": lasso.alpha_,
    }
    n_nonzero = int((np.abs(lasso.coef_) > 1e-8).sum())
    print(f"  {setup_name:10s}: SFA pair-diff variance = {eigval:.4f}, "
          f"selected alpha = {lasso.alpha_:.6f}, "
          f"Lasso non-zero coefs = {n_nonzero}/{n_bank}")


# ── Display: top coefficients per setup ───────────────────────────────────────
TOP_N_COEFS = 10

fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
axes = axes.flatten()

for ax, setup_name in zip(axes, LASSO_SETUPS):
    coefs = lasso_models[setup_name]["coefs"]
    order = np.argsort(-np.abs(coefs))[:TOP_N_COEFS]
    top_names  = [bank_columns[i] for i in order]
    top_values = coefs[order]

    colors = ["#1a7abf" if v >= 0 else "#c0392b" for v in top_values]
    ax.barh(range(len(top_values))[::-1], top_values, color=colors, alpha=0.88)
    ax.set_yticks(range(len(top_values))[::-1])
    ax.set_yticklabels(top_names, fontsize=8)
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set_title(f"{setup_name} — top {TOP_N_COEFS} Lasso coefficients", fontsize=11, fontweight="bold")
    ax.set_xlabel("Coefficient value")
    ax.grid(axis="x", alpha=0.3)

fig.suptitle(
    "Global Invariant Search (LASSO) — largest-magnitude coefficients per setup\n"
    "g(x) = Σ coef_i × bank_feature_i(x), fitted to reconstruct the SFA-optimal invariant direction",
    fontsize=13,
)
plt.show()


In [ ]:
# ── 6-1  Global Invariant Search — Symbolic regression (gplearn) ─────────────
# For each of the 4 setups, search for a closed-form expression g, built only
# from the 15 raw input variables (mean/std/q10/q90 each -> 60 candidates) and
# the operators +, -, *, /, log, sqrt, that stays approximately constant
# across that setup's pairs while varying meaningfully across the population:
#     Score(g) = Var(g(x)) / (L_pair(g) + eps)
# maximised via genetic programming (gplearn). The complexity/parsimony term
# from the original formula (Var / (L_pair + lambda*Complexity)) can't be
# folded into gplearn's custom fitness function directly — make_fitness() only
# exposes (y, y_pred, sample_weight), not program complexity — so instead we
# rely on gplearn's native `parsimony_coefficient`, which subtracts
# lambda * program_length from the raw fitness (a different but equivalent-in-
# spirit way of penalising complex expressions). Agreed with the user.

import sklearn.utils.validation as _skv
from gplearn.genetic import SymbolicRegressor
from gplearn.fitness import make_fitness

def _raw_input_summary_full(climate):
    """7-stat summary vector for all 15 raw input variables (all stats)."""
    aug = augmented_features_by_climate[climate]
    if climate == "historical":
        aug = aug[ae_split_indices["historical"]["test"]]
    X_flat = aug[:, :_n_raw_cols]
    return _compute_summary_stats(X_flat, n_input_variables, grid_points_per_patch, n_lat, n_lon)

raw_hist_summary_full        = _raw_input_summary_full("historical")        # (n_avail_hist, 15, 7)
raw_hist_summary_full_anchor = raw_hist_summary_full[anchor_pos]            # (N_PAIRS, 15, 7)
raw_ssp_summary_full_by_clim = {c: _raw_input_summary_full(c) for c in ssp_climates}

# Compatibility shim: gplearn 0.4.2 predates sklearn's removal of
# BaseEstimator._validate_data (sklearn >=1.6 uses a standalone function
# instead). Without this, SymbolicRegressor.fit() raises AttributeError.
if not hasattr(SymbolicRegressor, "_validate_data"):
    SymbolicRegressor._validate_data = (
        lambda self, X="no_validation", y="no_validation", **kw: _skv.validate_data(self, X, y, **kw)
    )

SYMREG_STATS = ["mean"]
symreg_stat_indices = [SUMMARY_STAT_NAMES.index(s) for s in SYMREG_STATS]

SYMREG_POPULATION_SIZE       = 2000
SYMREG_GENERATIONS           = 20
SYMREG_PARSIMONY_COEFFICIENT = 0.001
SYMREG_EPS                   = 1e-6   # denominator floor, avoids blow-up when L_pair -> 0

symreg_bank_columns = [f"{var}__{stat}" for var in selected_variables for stat in SYMREG_STATS]


def sfa_score(y, y_pred, sample_weight):
    """Score(g) = Var(g(x)) / (L_pair(g) + eps). Robust to gplearn's internal
    validation calls, which evaluate this on tiny dummy arrays — the pair
    split is always the first/second half of whatever y_pred it is given."""
    half   = len(y_pred) // 2
    g_hist = y_pred[:half]
    g_ssp  = y_pred[half:2 * half]
    l_pair = np.mean((g_hist - g_ssp) ** 2)
    var_g  = np.var(y_pred)
    return var_g / (l_pair + SYMREG_EPS)


sfa_fitness = make_fitness(function=sfa_score, greater_is_better=True, wrap=False)

symreg_models = {}   # {setup_name: fitted SymbolicRegressor}

for setup_name in LASSO_SETUPS:
    target_climate, ssp_index = pair_results[setup_name]

    hist_raw_summary = raw_hist_summary_full_anchor[:, :, symreg_stat_indices].reshape(N_PAIRS, -1)
    ssp_raw_summary = np.stack([
        raw_ssp_summary_full_by_clim[c][i] for c, i in zip(target_climate, ssp_index)
    ])[:, :, symreg_stat_indices].reshape(N_PAIRS, -1)

    X_pooled = np.vstack([hist_raw_summary, ssp_raw_summary])          # (2*N_PAIRS, 60)
    X_std = X_pooled.std(axis=0)
    X_std_safe = np.where(X_std > 0, X_std, 1.0)
    X_scaled = (X_pooled - X_pooled.mean(axis=0)) / X_std_safe
    y_dummy = np.zeros(len(X_scaled))   # unused by sfa_score, required by the sklearn API

    print(f"Running symbolic regression for {setup_name}…")
    est = SymbolicRegressor(
        population_size=SYMREG_POPULATION_SIZE,
        generations=SYMREG_GENERATIONS,
        function_set=("add", "sub", "mul", "div", "log", "sqrt"),
        metric=sfa_fitness,
        parsimony_coefficient=SYMREG_PARSIMONY_COEFFICIENT,
        feature_names=symreg_bank_columns,   # print real variable names instead of X0..X59
        random_state=random_seed,
        n_jobs=-1,
        verbose=0,
    )
    est.fit(X_scaled, y_dummy)
    symreg_models[setup_name] = est

    print(f"  {setup_name:10s}: score={est._program.raw_fitness_:.4f}  formula: {est._program}")

PAIR_SETUP_COLORS = {
    "Random":   "#B0B0B0",
    "RawData":  "#16A085",
    "CERA":     "#5B8DB8",
    "Baseline": "#E07B39",
    "ClimaX":   "#9B59B6",
}

# ── Display: score comparison + readable formula cards ───────────────────────
fig = plt.figure(figsize=(14, 10), constrained_layout=True)
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.6])

ax_score = fig.add_subplot(gs[0])
scores = [symreg_models[s]._program.raw_fitness_ for s in LASSO_SETUPS]
bar_colors = [PAIR_SETUP_COLORS.get(s, "#5B8DB8") for s in LASSO_SETUPS]
ax_score.bar(LASSO_SETUPS, scores, color=bar_colors, alpha=0.88)
ax_score.set_ylabel("Score(g) = Var(g) / (L_pair + eps)")
ax_score.set_title(
    "Symbolic regression score per setup\n(higher = more invariant across pairs and more variable across the climate ensemble)",
    fontsize=12, fontweight="bold",
)
ax_score.grid(axis="y", alpha=0.3)

ax_text = fig.add_subplot(gs[1])
ax_text.axis("off")
n_setups = len(LASSO_SETUPS)
block_h = 1.0 / n_setups
for i, setup_name in enumerate(LASSO_SETUPS):
    y_top = 1 - i * block_h
    color = PAIR_SETUP_COLORS.get(setup_name, "black")
    formula_str = str(symreg_models[setup_name]._program)
    score = symreg_models[setup_name]._program.raw_fitness_

    ax_text.text(0.01, y_top - 0.03, f"{setup_name}   (score = {score:.3f})",
                 fontsize=13, fontweight="bold", color=color,
                 transform=ax_text.transAxes, va="top")
    ax_text.text(0.03, y_top - 0.12, f"g(x) = {formula_str}",
                 fontsize=10, family="monospace", wrap=True,
                 transform=ax_text.transAxes, va="top")
    if i < n_setups - 1:
        ax_text.plot([0, 1], [y_top - block_h + 0.02] * 2,
                     transform=ax_text.transAxes, color="lightgray", linewidth=0.8)

ax_text.set_title("Discovered invariant formulas per setup", fontsize=12, fontweight="bold")
fig.suptitle("Global Invariant Search — Symbolic Regression results", fontsize=14)
plt.show()


In [ ]:
# ── 6-2  Symbolic regression — diversified top formulas (SymbolicTransformer) ─
# Same objective/bank/fitness as cell 6-1 (SFA-style Score(g) = Var(g)/(L_pair+eps),
# 15 raw variables x mean/std/q10/q90, operators +,-,*,/,log,sqrt), but using
# gplearn's SymbolicTransformer instead of SymbolicRegressor: rather than
# keeping only the single best individual, its "hall of fame" mechanism picks
# HOF_N_COMPONENTS formulas that are simultaneously high-scoring AND mutually
# decorrelated (near-duplicate formulas are discarded), giving a richer,
# non-redundant set of candidate invariants per setup instead of just one.

import sklearn.utils.validation as _skv
from gplearn.genetic import SymbolicTransformer

# Same compatibility shim as cell 6-1 (gplearn 0.4.2 predates sklearn's removal
# of BaseEstimator._validate_data), applied separately here: SymbolicTransformer
# is a sibling class of SymbolicRegressor (both inherit from BaseSymbolic), so
# the earlier patch does not carry over automatically.
if not hasattr(SymbolicTransformer, "_validate_data"):
    SymbolicTransformer._validate_data = (
        lambda self, X="no_validation", y="no_validation", **kw: _skv.validate_data(self, X, y, **kw)
    )

HOF_POOL_SIZE    = 50   # candidate pool size before decorrelation (gplearn's `hall_of_fame`)
HOF_N_COMPONENTS = 10   # final number of (decorrelated) formulas kept per setup

symtransform_models = {}   # {setup_name: fitted SymbolicTransformer}

for setup_name in LASSO_SETUPS:
    target_climate, ssp_index = pair_results[setup_name]

    hist_raw_summary = raw_hist_summary_full_anchor[:, :, symreg_stat_indices].reshape(N_PAIRS, -1)
    ssp_raw_summary = np.stack([
        raw_ssp_summary_full_by_clim[c][i] for c, i in zip(target_climate, ssp_index)
    ])[:, :, symreg_stat_indices].reshape(N_PAIRS, -1)

    X_pooled = np.vstack([hist_raw_summary, ssp_raw_summary])
    X_std = X_pooled.std(axis=0)
    X_std_safe = np.where(X_std > 0, X_std, 1.0)
    X_scaled = (X_pooled - X_pooled.mean(axis=0)) / X_std_safe
    y_dummy = np.zeros(len(X_scaled))   # unused by sfa_score, required by the sklearn API

    print(f"Running SymbolicTransformer for {setup_name}…")
    est = SymbolicTransformer(
        hall_of_fame=HOF_POOL_SIZE,
        n_components=HOF_N_COMPONENTS,
        population_size=SYMREG_POPULATION_SIZE,
        generations=SYMREG_GENERATIONS,
        function_set=("add", "sub", "mul", "div", "log", "sqrt"),
        metric=sfa_fitness,   # reuse the same fitness object built in cell 6-1
        parsimony_coefficient=SYMREG_PARSIMONY_COEFFICIENT,
        feature_names=symreg_bank_columns,
        random_state=random_seed,
        n_jobs=-1,
        verbose=0,
    )
    est.fit(X_scaled, y_dummy)
    symtransform_models[setup_name] = est

    scores = [p.raw_fitness_ for p in est._best_programs]
    print(f"  {setup_name:10s}: {len(est._best_programs)} formulas, "
          f"score range [{min(scores):.3f}, {max(scores):.3f}]")


# ── Display: ranked, decorrelated formulas per setup ──────────────────────────
for setup_name in LASSO_SETUPS:
    ranked = sorted(symtransform_models[setup_name]._best_programs,
                     key=lambda p: p.raw_fitness_, reverse=True)
    scores = [p.raw_fitness_ for p in ranked]
    n = len(ranked)

    fig = plt.figure(figsize=(15, max(6, n)), constrained_layout=True)
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.8])

    ax_bar = fig.add_subplot(gs[0])
    y_pos = np.arange(n)[::-1]
    ax_bar.barh(y_pos, scores, color=PAIR_SETUP_COLORS.get(setup_name, "#5B8DB8"), alpha=0.88)
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels([f"#{i + 1}" for i in range(n)], fontsize=9)
    ax_bar.set_xlabel("Score(g)")
    ax_bar.set_title("Rank", fontsize=10)
    ax_bar.grid(axis="x", alpha=0.3)

    ax_text = fig.add_subplot(gs[1])
    ax_text.axis("off")
    for i, program in enumerate(ranked):
        y_top = 1 - i / n
        ax_text.text(0.0, y_top - 0.01, f"#{i + 1}  (score = {program.raw_fitness_:.3f})",
                     fontsize=9, fontweight="bold", transform=ax_text.transAxes, va="top")
        ax_text.text(0.02, y_top - 0.05, str(program),
                     fontsize=8, family="monospace", wrap=True,
                     transform=ax_text.transAxes, va="top")

    fig.suptitle(
        f"{setup_name} — top {HOF_N_COMPONENTS} decorrelated invariant formulas (SymbolicTransformer)",
        fontsize=13, fontweight="bold",
    )
    plt.show()


In [ ]:
# ── 6-2  Symbolic regression — diversified top formulas (SymbolicTransformer) ─
# Same objective/bank/fitness as cell 6-1 (SFA-style Score(g) = Var(g)/(L_pair+eps),
# 15 raw variables x mean/std/q10/q90, operators +,-,*,/,log,sqrt), using
# gplearn's SymbolicTransformer instead of SymbolicRegressor: rather than
# keeping only the single best individual, its "hall of fame" mechanism picks
# HOF_N_COMPONENTS formulas that are simultaneously high-scoring AND mutually
# decorrelated (near-duplicate formulas are discarded), giving a richer,
# non-redundant set of candidate invariants per setup instead of just one.
#
# Difference vs the original version: the 15 input variables are fed in their
# native physical units (K, kg/kg, m/s, Pa, ...) — read directly from
# features_by_climate, NOT from the already-standardized X_sc used elsewhere
# in the notebook — and are NOT re-standardized before fitting. `exp` was
# considered but deliberately left out of function_set: on unnormalized inputs
# (psl ~1e5, temperatures ~3e2) it overflows almost immediately and the
# resulting NaN fitness just wastes search budget without discovering
# anything useful.

import sklearn.utils.validation as _skv
from gplearn.genetic import SymbolicTransformer

LASSO_SETUPS = ["CERA", "Baseline", "ClimaX"]

# Same compatibility shim as cell 6-1 (gplearn 0.4.2 predates sklearn's removal
# of BaseEstimator._validate_data), applied separately here: SymbolicTransformer
# is a sibling class of SymbolicRegressor (both inherit from BaseSymbolic), so
# the earlier patch does not carry over automatically.
if not hasattr(SymbolicTransformer, "_validate_data"):
    SymbolicTransformer._validate_data = (
        lambda self, X="no_validation", y="no_validation", **kw: _skv.validate_data(self, X, y, **kw)
    )

HOF_POOL_SIZE    = 50   # candidate pool size before decorrelation (gplearn's `hall_of_fame`)
HOF_N_COMPONENTS = 10   # final number of (decorrelated) formulas kept per setup


# ── Raw input variables in native physical units (no standardization) ────────
def _raw_input_summary_physical(climate):
    """7-stat summary vector for the 15 raw input variables, in native
    physical units — reads features_by_climate directly, bypassing the
    standardized X_sc block used elsewhere in the notebook."""
    X_raw = np.asarray(features_by_climate[climate], dtype=np.float64)
    if climate == "historical":
        X_raw = X_raw[ae_split_indices["historical"]["test"]]
    return _compute_summary_stats(X_raw, n_input_variables, grid_points_per_patch, n_lat, n_lon)


raw_hist_summary_physical_full        = _raw_input_summary_physical("historical")
raw_hist_summary_physical_full_anchor = raw_hist_summary_physical_full[anchor_pos]
raw_ssp_summary_physical_full_by_clim = {c: _raw_input_summary_physical(c) for c in ssp_climates}

symtransform_models = {}   # {setup_name: fitted SymbolicTransformer}

for setup_name in LASSO_SETUPS:
    target_climate, ssp_index = pair_results[setup_name]

    hist_raw_summary = raw_hist_summary_physical_full_anchor[:, :, symreg_stat_indices].reshape(N_PAIRS, -1)
    ssp_raw_summary = np.stack([
        raw_ssp_summary_physical_full_by_clim[c][i] for c, i in zip(target_climate, ssp_index)
    ])[:, :, symreg_stat_indices].reshape(N_PAIRS, -1)

    X_pooled = np.vstack([hist_raw_summary, ssp_raw_summary])   # native physical units, unscaled
    y_dummy  = np.zeros(len(X_pooled))   # unused by sfa_score, required by the sklearn API

    print(f"Running SymbolicTransformer for {setup_name}…")
    est = SymbolicTransformer(
        hall_of_fame=HOF_POOL_SIZE,
        n_components=HOF_N_COMPONENTS,
        population_size=SYMREG_POPULATION_SIZE,
        generations=SYMREG_GENERATIONS,
        function_set=("add", "sub", "mul", "div", "log", "sqrt"),
        metric=sfa_fitness,   # reuse the same fitness object built in cell 6-1
        parsimony_coefficient=SYMREG_PARSIMONY_COEFFICIENT,
        feature_names=symreg_bank_columns,
        random_state=random_seed,
        n_jobs=-1,
        verbose=0,
    )
    est.fit(X_pooled, y_dummy)
    symtransform_models[setup_name] = est

    scores = [p.raw_fitness_ for p in est._best_programs]
    print(f"  {setup_name:10s}: {len(est._best_programs)} formulas, "
          f"score range [{min(scores):.3f}, {max(scores):.3f}]")


# ── Display: ranked, decorrelated formulas per setup ──────────────────────────
for setup_name in LASSO_SETUPS:
    ranked = sorted(symtransform_models[setup_name]._best_programs,
                     key=lambda p: p.raw_fitness_, reverse=True)
    scores = [p.raw_fitness_ for p in ranked]
    n = len(ranked)

    fig = plt.figure(figsize=(15, max(6, n)), constrained_layout=True)
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.8])

    ax_bar = fig.add_subplot(gs[0])
    y_pos = np.arange(n)[::-1]
    ax_bar.barh(y_pos, scores, color=PAIR_SETUP_COLORS.get(setup_name, "#5B8DB8"), alpha=0.88)
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels([f"#{i + 1}" for i in range(n)], fontsize=9)
    ax_bar.set_xlabel("Score(g)")
    ax_bar.set_title("Rank", fontsize=10)
    ax_bar.grid(axis="x", alpha=0.3)

    ax_text = fig.add_subplot(gs[1])
    ax_text.axis("off")
    for i, program in enumerate(ranked):
        y_top = 1 - i / n
        ax_text.text(0.0, y_top - 0.01, f"#{i + 1}  (score = {program.raw_fitness_:.3f})",
                     fontsize=9, fontweight="bold", transform=ax_text.transAxes, va="top")
        ax_text.text(0.02, y_top - 0.05, str(program),
                     fontsize=8, family="monospace", wrap=True,
                     transform=ax_text.transAxes, va="top")

    fig.suptitle(
        f"{setup_name} — top {HOF_N_COMPONENTS} decorrelated invariant formulas (SymbolicTransformer)",
        fontsize=13, fontweight="bold",
    )
    plt.show()


## Part 7 - Invariant subspaces within the latent

Sections 5 and 6 searched for a single scalar invariant `g(x)`, either along a hand-built bank of physical quantities (LASSO/SFA, cell 6-0) or via symbolic regression on the 15 raw variables (cells 6-1/6-2), using nearest-neighbour pairs found by a full-space Euclidean search in each setup's latent (48D for CERA/Baseline, 64D for ClimaX).

That full-space pairing can dilute or hide invariants that the network encodes in a specific sub-space of the latent rather than across all coordinates at once: if dimension *k* cleanly encodes some invariant but has small variance relative to other dimensions, the Euclidean NN search is dominated by the other dimensions and ends up matching points that are *not* particularly close along dimension *k* — so a downstream invariant search restricted to those pairs will under-estimate (or miss entirely) whatever *k* encodes.

This section looks for such invariant sub-spaces directly inside each setup's own latent coordinates, using the *existing* nearest-neighbour pairs (`pair_results`) already computed in Part 5. For a candidate direction `v` in latent space, define `g(x) = v . Z(x)` and the same SFA-style objective as before:

$$
\text{score}(v) = \frac{\mathrm{Var}(g)}{L_{pair}(g) + \varepsilon}, \qquad L_{pair}(g) = \mathbb{E}\big[(g(x_{hist}) - g(x_{ssp}))^2\big]
$$

Maximising this ratio over all directions `v` is a generalized Rayleigh quotient problem, solved exactly (no search, no gplearn) by the generalized eigenvalue problem `M_pair v = lambda * M_pop v`, where `M_pair` is the (uncentered) second moment of `Z_hist - Z_ssp` and `M_pop` is the covariance of the pooled population — the same linear algebra as the LASSO/SFA step in cell 6-0, applied here to the latent's own 48/64 coordinates instead of the hand-built feature bank.

The resulting eigenvectors are `M_pop`-orthonormal (statistically decorrelated), and eigenvalues that cluster tightly together indicate a genuinely multi-dimensional invariant sub-space rather than a single privileged axis: for two `M_pop`-orthonormal eigenvectors with eigenvalues lambda_1, lambda_2, the score of *any* combination `a.v_1 + b.v_2` is the weighted average `(a^2 lambda_1 + b^2 lambda_2)/(a^2+b^2)` — so when lambda_1 is approximately lambda_2, the *entire* plane they span is an equally good invariant candidate, not just the two specific (largely arbitrary) axes the solver happens to return. Sub-spaces are therefore detected as clusters of nearby eigenvalues, separated by a gap from the rest of the spectrum, and analysed jointly (not one axis at a time) in Part 8.

Run for the 3 latent-based setups: **CERA, Baseline, ClimaX**.


In [ ]:
# ── 7-1  Helper: reconstruct Z_hist/Z_ssp for the EXISTING (full-space) pairing ─
# Mirrors the internals of latent_pair_distances (cell 5-7), but returns the
# latent matrices themselves instead of a distance, so we can build M_pair/M_pop
# from them below.

from scipy.linalg import eigh

def get_latent_hist_ssp(setup_name):
    '''Z_hist, Z_ssp: (N_PAIRS, dim) each, using the SAME pairing already stored
    in pair_results[setup_name] (full-space Euclidean NN, computed in Part 5).'''
    latent_key, slice_dim = PAIR_LATENT_KEYS[setup_name]
    target_climate, ssp_index = pair_results[setup_name]

    hist_lmeta = pd.DataFrame(latent_test_sets[latent_key]["latent_test_metadata_by_climate"]["historical"])
    hist_lkeys = _sample_keys(hist_lmeta)
    key_to_row = {k: i for i, k in enumerate(hist_lkeys)}
    hist_rows  = np.array([key_to_row[k] for k in anchor_keys])
    Z_hist_all = np.asarray(latent_test_sets[latent_key]["latent_test_by_climate"]["historical"])
    Z_hist     = Z_hist_all[hist_rows]

    Z_ssp = np.empty_like(Z_hist)
    for c in np.unique(target_climate):
        mask = target_climate == c
        raw_key_to_row = {k: i for i, k in enumerate(_sample_keys(metadata_by_climate[c]))}
        lmeta_c = pd.DataFrame(latent_test_sets[latent_key]["latent_test_metadata_by_climate"][c])
        keys_c  = _sample_keys(lmeta_c)
        raw_row_to_latent_row = {raw_key_to_row[k]: i for i, k in enumerate(keys_c)}
        Zc_all = np.asarray(latent_test_sets[latent_key]["latent_test_by_climate"][c])
        latent_rows = np.array([raw_row_to_latent_row[i] for i in ssp_index[mask]])
        Z_ssp[mask] = Zc_all[latent_rows]

    if slice_dim is not None:
        Z_hist = Z_hist[:, :slice_dim]
        Z_ssp  = Z_ssp[:, :slice_dim]

    return Z_hist, Z_ssp


SUBSPACE_SETUPS = ["CERA", "Baseline", "ClimaX"]
print("Helper ready. Will analyse:", SUBSPACE_SETUPS)


In [ ]:
# ── 7-2  Train/holdout split of the N_PAIRS anchors (shared across setups) ────
# Same discipline as the earlier blind-search experiments: fit M_pair/M_pop and
# find eigen-directions on TRAIN only, validate on HOLDOUT before trusting a
# sub-space. Anchor position i means the same physical historical sample in
# every setup, since all setups share the same anchor_pos/anchor_keys.

EIGEN_SPLIT_SEED       = 0
EIGEN_HOLDOUT_FRACTION = 0.3

_eigen_rng  = np.random.default_rng(EIGEN_SPLIT_SEED)
_eigen_perm = _eigen_rng.permutation(N_PAIRS)
_n_holdout  = int(N_PAIRS * EIGEN_HOLDOUT_FRACTION)
EIGEN_HOLDOUT_IDX = _eigen_perm[:_n_holdout]
EIGEN_TRAIN_IDX   = _eigen_perm[_n_holdout:]

print(f"N_PAIRS={N_PAIRS}  ->  train={len(EIGEN_TRAIN_IDX)}, holdout={len(EIGEN_HOLDOUT_IDX)}")


In [ ]:
# ── 7-3  M_pair, M_pop and generalized eigenvalue decomposition, per setup ────
# score(v) = Var(v.Z) / L_pair(v.Z) is a generalized Rayleigh quotient; its
# stationary points are given by M_pair @ v = lambda * M_pop @ v (scipy.linalg.eigh
# on two symmetric matrices), exactly the same math as the LASSO/SFA step in
# cell 6-0, applied here to the latent's own coordinates instead of the hand-built
# feature bank. Eigenvectors come back M_pop-orthonormal, i.e. Var(v_i.Z)=1 for
# each of them and Cov(v_i.Z, v_j.Z)=0 for i!=j -> lambda_i = L_pair(v_i) exactly,
# so score(v_i) = 1/lambda_i. Fit on TRAIN only; validated on HOLDOUT in 7-6.

EIGEN_RIDGE_EPS = 1e-6   # numerical stabiliser for M_pop, same role as RIDGE_EPS in cell 6-0

latent_eigen_results = {}   # {setup_name: {"eigvals","eigvecs","mean","std","dim"}}

for setup_name in SUBSPACE_SETUPS:
    Z_hist, Z_ssp = get_latent_hist_ssp(setup_name)
    Zh_tr, Zs_tr  = Z_hist[EIGEN_TRAIN_IDX], Z_ssp[EIGEN_TRAIN_IDX]

    pool_tr  = np.vstack([Zh_tr, Zs_tr])
    mean     = pool_tr.mean(axis=0)
    std      = pool_tr.std(axis=0)
    std_safe = np.where(std > 0, std, 1.0)

    Zh_std = (Zh_tr - mean) / std_safe
    Zs_std = (Zs_tr - mean) / std_safe

    D = Zh_std - Zs_std
    M_pair = D.T @ D / len(D)

    pool_std = np.vstack([Zh_std, Zs_std])
    M_pop = np.cov(pool_std, rowvar=False)
    dim = M_pop.shape[0]
    M_pop_reg = M_pop + EIGEN_RIDGE_EPS * np.eye(dim)

    eigvals, eigvecs = eigh(M_pair, M_pop_reg)   # ascending eigvals

    latent_eigen_results[setup_name] = {
        "eigvals": eigvals, "eigvecs": eigvecs, "mean": mean, "std": std_safe, "dim": dim,
    }
    best_scores = 1.0 / np.clip(eigvals[:5], 1e-8, None)
    print(f"{setup_name:10s}: dim={dim:3d}  smallest 5 eigenvalues={np.round(eigvals[:5], 5)}  "
          f"-> best scores (1/lambda)={np.round(best_scores, 2)}")


In [ ]:
# ── 7-4  Spectrum plot: score (1/lambda) for the smallest eigenvalues, per setup ─

N_SHOW = 20

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
axes = axes.flatten()

for ax, setup_name in zip(axes, SUBSPACE_SETUPS):
    eigvals = latent_eigen_results[setup_name]["eigvals"]
    n_show  = min(N_SHOW, len(eigvals))
    scores  = 1.0 / np.clip(eigvals[:n_show], 1e-8, None)

    ax.bar(np.arange(n_show), scores, color=PAIR_SETUP_COLORS.get(setup_name, "#5B8DB8"), alpha=0.88)
    ax.set_yscale("log")
    ax.set_xlabel("Eigenvector rank (smallest lambda first)")
    ax.set_ylabel("score = 1/lambda  (log scale)")
    ax.set_title(f"{setup_name} — latent invariant-direction spectrum", fontsize=11, fontweight="bold")
    ax.grid(axis="y", alpha=0.3, which="both")

fig.suptitle(
    "Generalized-eigenvalue spectrum of Var(v.Z)/L_pair(v.Z), per setup (TRAIN split)\n"
    "Look for gaps: a cluster of nearby bars, well separated from the rest, is a candidate invariant sub-space",
    fontsize=12,
)
plt.show()


In [ ]:
# ── 7-5  Automatic gap-detection clustering of the eigenvalue spectrum ────────
# Walk the ascending eigenvalues; a "gap" is a jump in consecutive eigenvalues
# by more than GAP_RATIO_THRESHOLD. Each run of eigenvalues between two gaps is
# a candidate invariant sub-space (dimension = number of eigenvalues in the
# run) — no manual tuning per setup, purely threshold-driven. Up to
# MAX_SUBSPACES clusters are kept per setup, no cap on how many dimensions a
# single cluster may contain (kept at its natural size).

GAP_RATIO_THRESHOLD = 2.0
MAX_CONSIDER        = 20
MAX_SUBSPACES        = 3


def cluster_eigenvalues(eigvals, max_clusters=MAX_SUBSPACES, max_consider=MAX_CONSIDER,
                         gap_ratio_threshold=GAP_RATIO_THRESHOLD, eps_floor=1e-8):
    ev = np.clip(np.asarray(eigvals), eps_floor, None)
    n = min(max_consider, len(ev))
    clusters = []
    current = [0]
    for i in range(1, n):
        if ev[i] / ev[i - 1] > gap_ratio_threshold:
            clusters.append(current)
            if len(clusters) == max_clusters:
                return clusters
            current = [i]
        else:
            current.append(i)
    clusters.append(current)
    return clusters[:max_clusters]


subspace_clusters = {}   # {setup_name: [[idx,...], [idx,...], ...]}  up to 3 lists

for setup_name in SUBSPACE_SETUPS:
    eigvals = latent_eigen_results[setup_name]["eigvals"]
    clusters = cluster_eigenvalues(eigvals)
    subspace_clusters[setup_name] = clusters
    print(f"\n{setup_name}: {len(clusters)} sub-space(s) found (gap ratio > {GAP_RATIO_THRESHOLD}):")
    for k, idxs in enumerate(clusters):
        scores = 1.0 / np.clip(eigvals[idxs], 1e-8, None)
        print(f"   sub-space {k}: dim={len(idxs)}  eigenvector indices={idxs}  scores={np.round(scores, 2)}")


In [ ]:
# ── 7-6  Holdout validation of each detected sub-space ─────────────────────────
# For each eigenvector kept in 7-5, recompute its score on the HOLDOUT split
# (same v, unseen pairs) to check the direction is real and not a TRAIN-only
# numerical artefact. Report per-eigenvector holdout score plus each
# sub-space's aggregate (mean over its eigenvectors).

def eigvec_score_on_holdout(setup_name, v):
    Z_hist, Z_ssp = get_latent_hist_ssp(setup_name)
    mean = latent_eigen_results[setup_name]["mean"]
    std  = latent_eigen_results[setup_name]["std"]

    Zh = (Z_hist[EIGEN_HOLDOUT_IDX] - mean) / std
    Zs = (Z_ssp[EIGEN_HOLDOUT_IDX]  - mean) / std

    gh, gs = Zh @ v, Zs @ v
    l_pair = np.mean((gh - gs) ** 2)
    var_g  = np.var(np.concatenate([gh, gs]))
    return var_g / (l_pair + 1e-9)


subspace_holdout_scores = {}   # {setup_name: [array of per-eigvec holdout scores, ...]}

for setup_name in SUBSPACE_SETUPS:
    eigvecs = latent_eigen_results[setup_name]["eigvecs"]
    eigvals = latent_eigen_results[setup_name]["eigvals"]
    print(f"\n{setup_name}:")
    per_setup_holdout = []
    for k, idxs in enumerate(subspace_clusters[setup_name]):
        train_scores   = 1.0 / np.clip(eigvals[idxs], 1e-8, None)
        holdout_scores = np.array([eigvec_score_on_holdout(setup_name, eigvecs[:, i]) for i in idxs])
        per_setup_holdout.append(holdout_scores)
        ratio = holdout_scores / np.clip(train_scores, 1e-8, None)
        print(f"   sub-space {k}: train scores={np.round(train_scores,2)}  "
              f"holdout scores={np.round(holdout_scores,2)}  holdout/train ratio={np.round(ratio,2)}")
    subspace_holdout_scores[setup_name] = per_setup_holdout


In [ ]:
# ── 7-7  Interpretation: correlate each sub-space direction with known quantities ─
# For every kept eigenvector, project the pooled (hist anchors + their paired
# ssp points) population onto it and correlate against the 25 physical
# invariants + 15 raw variables (mean statistic, cells 5-0/6-2 style) -- a
# first, cheap interpretability read-out before running symbolic regression in
# Part 8.

mean_idx    = SUMMARY_STAT_NAMES.index("mean")
known_names = list(physical_variable_names) + list(selected_variables)

for setup_name in SUBSPACE_SETUPS:
    Z_hist, Z_ssp = get_latent_hist_ssp(setup_name)
    mean, std = latent_eigen_results[setup_name]["mean"], latent_eigen_results[setup_name]["std"]
    Zh_std = (Z_hist - mean) / std
    Zs_std = (Z_ssp  - mean) / std

    target_climate, ssp_index = pair_results[setup_name]
    ssp_inv = np.stack([ssp_invariant_summary_by_clim[c][i] for c, i in zip(target_climate, ssp_index)])
    ssp_raw = np.stack([raw_ssp_summary_physical_full_by_clim[c][i] for c, i in zip(target_climate, ssp_index)])

    known_hist = np.concatenate([hist_invariant_summary_anchor[:, :, mean_idx],
                                  raw_hist_summary_physical_full_anchor[:, :, mean_idx]], axis=1)
    known_ssp  = np.concatenate([ssp_inv[:, :, mean_idx], ssp_raw[:, :, mean_idx]], axis=1)
    known_pool = np.vstack([known_hist, known_ssp])   # (2*N_PAIRS, 40)

    row_labels, corr_rows = [], []
    for k, idxs in enumerate(subspace_clusters[setup_name]):
        for i in idxs:
            v = latent_eigen_results[setup_name]["eigvecs"][:, i]
            g_pool = np.concatenate([Zh_std @ v, Zs_std @ v])
            corr_rows.append([np.corrcoef(g_pool, known_pool[:, j])[0, 1] for j in range(known_pool.shape[1])])
            row_labels.append(f"sub{k}.v{i}")

    corr_matrix = np.array(corr_rows)
    fig, ax = plt.subplots(figsize=(14, 0.35 * len(row_labels) + 2), constrained_layout=True)
    im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(np.arange(len(known_names)))
    ax.set_xticklabels(known_names, rotation=90, fontsize=7)
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=8)
    plt.colorbar(im, ax=ax, label="Pearson correlation")
    ax.set_title(f"{setup_name} — correlation of each kept eigenvector with known quantities", fontsize=11)
    plt.show()


## Part 8 - Per-subspace symbolic regression

For each invariant sub-space found in Part 7 (up to 3 per setup), this section:

1. Re-derives a nearest-neighbour pairing restricted to that sub-space only (Euclidean distance computed after projecting onto the sub-space's eigenvectors — rotation-invariant within the sub-space, honouring the fact that individual eigenvectors inside a degenerate/near-degenerate cluster are not individually meaningful, only their span is).
2. Runs the same SFA-style symbolic regression (`gplearn.SymbolicTransformer`, same fitness `sfa_score`, cells 6-1/6-2) restricted to the exact same 15 raw variables in physical units, using this new sub-space-specific pairing instead of the global `pair_results`.
3. Reports the top 10 (decorrelated) candidate formulas per sub-space, validated on the same HOLDOUT split used throughout, and cross-checks the best formula's holdout score against the sub-space's own "pure latent" holdout score from Part 7 (7-6).

Each setup's sub-spaces are analysed using that setup's own latent — up to 4 setups x 3 sub-spaces = 12 independent symbolic-regression runs.


In [ ]:
# ── 8-1  Helper: full latent pool (hist anchors + ALL available ssp points) ───
# Mirrors the internals of build_latent_pairs (cell 5-1), but returns the pool
# matrices themselves instead of doing the full-space NN search — we need the
# raw pool to run our OWN sub-space-restricted NN search in 8-2.

def get_latent_hist_and_ssp_pool(setup_name):
    latent_key, slice_dim = PAIR_LATENT_KEYS[setup_name]

    hist_lmeta = pd.DataFrame(latent_test_sets[latent_key]["latent_test_metadata_by_climate"]["historical"])
    hist_lkeys = _sample_keys(hist_lmeta)
    key_to_row = {k: i for i, k in enumerate(hist_lkeys)}
    hist_rows  = np.array([key_to_row[k] for k in anchor_keys])
    Z_hist_all = np.asarray(latent_test_sets[latent_key]["latent_test_by_climate"]["historical"])
    Z_hist     = Z_hist_all[hist_rows]
    if slice_dim is not None:
        Z_hist = Z_hist[:, :slice_dim]

    Z_ssp_parts, ssp_pool_climate_parts, ssp_pool_index_parts = [], [], []
    for c in ssp_climates:
        Zc = np.asarray(latent_test_sets[latent_key]["latent_test_by_climate"][c])
        if slice_dim is not None:
            Zc = Zc[:, :slice_dim]
        lmeta_c        = pd.DataFrame(latent_test_sets[latent_key]["latent_test_metadata_by_climate"][c])
        keys_c         = _sample_keys(lmeta_c)
        raw_key_to_row = {k: i for i, k in enumerate(_sample_keys(metadata_by_climate[c]))}
        idx_in_raw     = np.array([raw_key_to_row[k] for k in keys_c])

        Z_ssp_parts.append(Zc)
        ssp_pool_climate_parts.append(np.full(len(Zc), c))
        ssp_pool_index_parts.append(idx_in_raw)

    Z_ssp_pool       = np.vstack(Z_ssp_parts)
    ssp_pool_climate = np.concatenate(ssp_pool_climate_parts)
    ssp_pool_index   = np.concatenate(ssp_pool_index_parts)

    return Z_hist, Z_ssp_pool, ssp_pool_climate, ssp_pool_index


In [ ]:
# ── 8-2  Sub-space-restricted nearest-neighbour pairing ───────────────────────
# For a given (setup, sub-space) pair, project the hist anchors and the FULL
# ssp pool onto the sub-space's eigenvectors, then run a Euclidean NN search in
# that (small) projected space only -- this is rotation-invariant within the
# sub-space, matching the theoretical result that individual eigenvectors
# inside a near-degenerate cluster are not individually meaningful, only their
# span is.

from sklearn.neighbors import NearestNeighbors as _NN_subspace

def build_subspace_pairs(setup_name, cluster_idx):
    eig = latent_eigen_results[setup_name]
    v_sub = eig["eigvecs"][:, cluster_idx]          # (dim, k)
    mean, std = eig["mean"], eig["std"]

    Z_hist, Z_ssp_pool, ssp_pool_climate, ssp_pool_index = get_latent_hist_and_ssp_pool(setup_name)

    Zh_std = (Z_hist     - mean) / std
    Zp_std = (Z_ssp_pool - mean) / std

    hist_proj = Zh_std @ v_sub     # (N_PAIRS, k)
    pool_proj = Zp_std @ v_sub     # (n_pool, k)

    nn = _NN_subspace(n_neighbors=1).fit(pool_proj)
    _, nn_pos = nn.kneighbors(hist_proj)
    nn_pos = nn_pos[:, 0]

    return ssp_pool_climate[nn_pos], ssp_pool_index[nn_pos]


In [ ]:
# ── 8-3  Symbolic regression per (setup, sub-space) — up to 4 x 3 = 12 runs ───
# Same fitness (sfa_score), same 15-raw-variable physical-unit bank
# (raw_*_summary_physical_full_*, cell 6-2), same SymbolicTransformer settings
# as Part 6 -- only the pairing changes, to the sub-space-restricted one built
# in 8-2. Fit on TRAIN only; holdout re-scoring is done in 8-4.

subspace_pairs        = {}   # {(setup_name, k): (target_climate, ssp_index)}
subspace_symtransform = {}   # {(setup_name, k): fitted SymbolicTransformer}

for setup_name in SUBSPACE_SETUPS:
    for k, cluster_idx in enumerate(subspace_clusters[setup_name]):
        print(f"\nBuilding sub-space pairing + running SymbolicTransformer: {setup_name} / sub-space {k} "
              f"(dim={len(cluster_idx)})...")

        target_climate_sub, ssp_index_sub = build_subspace_pairs(setup_name, cluster_idx)
        subspace_pairs[(setup_name, k)] = (target_climate_sub, ssp_index_sub)

        hist_raw = raw_hist_summary_physical_full_anchor[:, :, symreg_stat_indices].reshape(N_PAIRS, -1)
        ssp_raw = np.stack([
            raw_ssp_summary_physical_full_by_clim[c][i] for c, i in zip(target_climate_sub, ssp_index_sub)
        ])[:, :, symreg_stat_indices].reshape(N_PAIRS, -1)

        hist_train = hist_raw[EIGEN_TRAIN_IDX]
        ssp_train  = ssp_raw[EIGEN_TRAIN_IDX]
        X_pooled   = np.vstack([hist_train, ssp_train])
        y_dummy    = np.zeros(len(X_pooled))

        est = SymbolicTransformer(
            hall_of_fame=HOF_POOL_SIZE,
            n_components=HOF_N_COMPONENTS,
            population_size=SYMREG_POPULATION_SIZE,
            generations=SYMREG_GENERATIONS,
            function_set=("add", "sub", "mul", "div", "log", "sqrt"),
            metric=sfa_fitness,
            parsimony_coefficient=SYMREG_PARSIMONY_COEFFICIENT,
            feature_names=symreg_bank_columns,
            random_state=random_seed,
            n_jobs=-1,
            verbose=0,
        )
        est.fit(X_pooled, y_dummy)
        subspace_symtransform[(setup_name, k)] = est

        scores = [p.raw_fitness_ for p in est._best_programs]
        print(f"  {setup_name} / sub-space {k}: {len(est._best_programs)} formulas, "
              f"train score range [{min(scores):.3f}, {max(scores):.3f}]")


In [ ]:
# ── 8-4  Holdout re-scoring of each sub-space's discovered formulas ───────────
# Same sfa_score convention (first half of y_pred = hist, second half = ssp),
# evaluated only on EIGEN_HOLDOUT_IDX -- exactly the same discipline used for
# the raw-variable searches in Part 6 and the eigen-directions in Part 7.

def score_program_on_holdout(program, target_climate_sub, ssp_index_sub):
    hist_raw = raw_hist_summary_physical_full_anchor[:, :, symreg_stat_indices].reshape(N_PAIRS, -1)
    ssp_raw = np.stack([
        raw_ssp_summary_physical_full_by_clim[c][i] for c, i in zip(target_climate_sub, ssp_index_sub)
    ])[:, :, symreg_stat_indices].reshape(N_PAIRS, -1)

    hist_hold = hist_raw[EIGEN_HOLDOUT_IDX]
    ssp_hold  = ssp_raw[EIGEN_HOLDOUT_IDX]
    X_hold = np.vstack([hist_hold, ssp_hold])

    y_pred = program.execute(X_hold)
    if not np.all(np.isfinite(y_pred)):
        return float("nan")
    half = len(y_pred) // 2
    g_hist, g_ssp = y_pred[:half], y_pred[half:2 * half]
    l_pair = np.mean((g_hist - g_ssp) ** 2)
    var_g  = np.var(y_pred)
    return var_g / (l_pair + 1e-6)


subspace_top10 = {}   # {(setup_name, k): list of dicts sorted by holdout score}

for setup_name in SUBSPACE_SETUPS:
    for k, cluster_idx in enumerate(subspace_clusters[setup_name]):
        est = subspace_symtransform[(setup_name, k)]
        target_climate_sub, ssp_index_sub = subspace_pairs[(setup_name, k)]

        rows = []
        for program in est._best_programs:
            train_score = program.raw_fitness_
            holdout_score = score_program_on_holdout(program, target_climate_sub, ssp_index_sub)
            rows.append({"expr": str(program), "train_score": train_score, "holdout_score": holdout_score})

        rows.sort(key=lambda r: (-r["holdout_score"] if np.isfinite(r["holdout_score"]) else float("inf")))
        subspace_top10[(setup_name, k)] = rows

        print(f"\n{setup_name} / sub-space {k} (dim={len(cluster_idx)}) — ranked by HOLDOUT score:")
        for r in rows:
            print(f"   holdout={r['holdout_score']:8.3f}  train={r['train_score']:8.3f}   {r['expr']}")


In [ ]:
# ── 8-5  Display: ranked formula cards per (setup, sub-space) ─────────────────

for setup_name in SUBSPACE_SETUPS:
    for k, cluster_idx in enumerate(subspace_clusters[setup_name]):
        rows = subspace_top10[(setup_name, k)]
        n = len(rows)

        fig = plt.figure(figsize=(15, max(6, n)), constrained_layout=True)
        gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.8])

        ax_bar = fig.add_subplot(gs[0])
        y_pos = np.arange(n)[::-1]
        holdout_scores = [r["holdout_score"] for r in rows]
        ax_bar.barh(y_pos, holdout_scores, color=PAIR_SETUP_COLORS.get(setup_name, "#5B8DB8"), alpha=0.88)
        ax_bar.set_yticks(y_pos)
        ax_bar.set_yticklabels([f"#{i + 1}" for i in range(n)], fontsize=9)
        ax_bar.set_xlabel("Holdout score")
        ax_bar.set_title("Rank (by holdout score)", fontsize=10)
        ax_bar.grid(axis="x", alpha=0.3)

        ax_text = fig.add_subplot(gs[1])
        ax_text.axis("off")
        for i, r in enumerate(rows):
            y_top = 1 - i / n
            ax_text.text(0.0, y_top - 0.01,
                         f"#{i + 1}  (holdout={r['holdout_score']:.3f}, train={r['train_score']:.3f})",
                         fontsize=9, fontweight="bold", transform=ax_text.transAxes, va="top")
            ax_text.text(0.02, y_top - 0.05, r["expr"], fontsize=8, family="monospace", wrap=True,
                         transform=ax_text.transAxes, va="top")

        fig.suptitle(
            f"{setup_name} — sub-space {k} (dim={len(cluster_idx)}) — top formulas (SymbolicTransformer)",
            fontsize=13, fontweight="bold",
        )
        plt.show()


In [ ]:
# ── 8-6  Cross-check: best raw-variable formula vs the sub-space's own score ──
# Compares the best gplearn holdout score (8-4) to the sub-space's aggregate
# "pure latent" holdout score (7-6, mean over its eigenvectors) -- close scores
# mean the 15 raw variables (mean statistic) can explain what this direction of
# the latent encodes; a much lower gplearn score means either gplearn hasn't
# found the right combination, or this direction needs information beyond
# patch-mean raw variables (std/gradients, or genuinely spatial structure).

print(f"{'setup':10s} {'subspace':9s} {'dim':4s} {'latent holdout (mean)':>22s} {'best formula holdout':>21s} {'ratio':>7s}")
for setup_name in SUBSPACE_SETUPS:
    for k, cluster_idx in enumerate(subspace_clusters[setup_name]):
        latent_score = float(np.mean(subspace_holdout_scores[setup_name][k]))
        best_formula_score = subspace_top10[(setup_name, k)][0]["holdout_score"]
        ratio = best_formula_score / latent_score if np.isfinite(best_formula_score) and latent_score > 0 else float("nan")
        print(f"{setup_name:10s} {k:<9d} {len(cluster_idx):<4d} {latent_score:22.3f} {best_formula_score:21.3f} {ratio:7.2f}")
